# Portfolio edition

Historical development notebook: ViSoBERT, six rule-based aspects and NMF. Requires the original checksum-locked master CSV, which is not distributed here. This is not the final seven-theme reporting pipeline.

Source code retained; cell outputs and execution metadata cleared for publication. See RUNNING.md for inputs, historical validation and limitations.


# MIS495 — Thy & Hậu | Sentiment, ABSA và Topic Modeling

## Goal

Notebook Google Colab này tái tạo kết quả của workbook `MIS495_Thy_Hậu_Sentiment.xlsx` từ **master review chính thức**, không đọc workbook kết quả làm đầu vào.

Đầu ra có đúng 6 sheet:

1. `Overall Sentiment`
2. `Aspect Summary`
3. `Aspect by Shop`
4. `Topic Definitions`
5. `Validation 100`
6. `Scored Reviews`

## Setup

- Mở notebook bằng một runtime mới: **Runtime → Disconnect and delete runtime**.
- Chọn **Runtime → Change runtime type → T4 GPU**.
- Chọn **Runtime → Run all**; không chạy cell lộn thứ tự.
- Khi được hỏi, upload đúng file `reviews_master_text_v1.csv` gồm 47.466 review.
- SHA-256 bắt buộc của input: `8dc7ea0a9b6999438c47f85eb210bfd6aa604662f8937f4b5d24b3daeef07c00`.
- Cell cuối phải báo `PASS` cho toàn bộ QA trước khi dùng file tải xuống.

## Context & Methods

- Overall sentiment: `5CD-AI/Vietnamese-Sentiment-visobert`, revision cố định `d7738b85200272f10ba8b048d80236c8c582b581`.
- Aspect sentiment: rule-based ABSA V1B cho 6 khía cạnh.
- Topic modeling: TF-IDF + NMF, `random_state = 495`, 10 topic.
- Topic names và thứ tự `Validation 100` là phần diễn giải/mẫu thủ công đã chốt cho đúng snapshot chính thức; notebook khai báo công khai các giá trị này trong code.

### Key Assumptions

- Kết quả chỉ tái lập chính xác khi dùng đúng CSV có checksum nêu trên.
- `Validation 100` hiện chứa nhãn mô hình, chưa phải ground truth người thật; không báo cáo accuracy hoặc macro-F1 trước khi gán nhãn độc lập.


### Nếu từng chạy bản cũ và đã gặp lỗi

Không tiếp tục trên kernel cũ. Hãy chọn `Runtime → Disconnect and delete runtime`, bật lại T4 rồi chạy bản STABLE V2 từ đầu. Bản này không thay `numpy`, `pandas` hoặc `matplotlib` của Colab.


## Cell 1 — Cài thư viện bổ sung, không thay thư viện lõi của Colab

In [ ]:
# Chỉ cài package bổ sung; giữ nguyên numpy/pandas/matplotlib của runtime Colab
import importlib.metadata as metadata
import subprocess
import sys

packages = [
    "transformers==4.57.6",
    "sentencepiece==0.2.2",
    "safetensors==0.8.0",
    "huggingface_hub==0.36.2",
    "openpyxl==3.1.5",
    "xlsxwriter==3.2.9",
    "scikit-learn==1.9.0",
    "tqdm==4.69.0",
]

print("Đang cài các package bổ sung...")
install_result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", *packages],
    text=True,
)
if install_result.returncode != 0:
    raise RuntimeError(
        "Cài thư viện thất bại. Hãy chọn Runtime → Disconnect and delete runtime, "
        "bật T4 GPU rồi Run all lại từ đầu."
    )

print("PASS: cài thư viện bổ sung thành công.")
for package in [
    "transformers", "torch", "pandas", "numpy", "sentencepiece",
    "openpyxl", "xlsxwriter", "scikit-learn", "tqdm", "matplotlib",
]:
    try:
        print(f"- {package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"- {package}: không tìm thấy")


## Cell 2 — Import, cấu hình và kiểm tra GPU

In [ ]:
# Import và cấu hình tái lập
import csv
import hashlib
import html
import io
import json
import math
import os
import platform
import random
import re
import time
import unicodedata
import zipfile

os.environ["TOKENIZERS_PARALLELISM"] = "false"
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import transformers
from tqdm.auto import tqdm
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    files = None
    IN_COLAB = False

RANDOM_SEED = 495
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
torch.use_deterministic_algorithms(True, warn_only=True)
if hasattr(torch.backends, 'cudnn'):
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

MODEL_ID = "5CD-AI/Vietnamese-Sentiment-visobert"
MODEL_REVISION = "d7738b85200272f10ba8b048d80236c8c582b581"
MAX_LENGTH = 256
BATCH_SIZE_GPU = 32
BATCH_SIZE_CPU = 8
LOW_CONFIDENCE_THRESHOLD = 0.60
TEXT_COLUMN = None  # None = tự tìm; hoặc đổi thành "review_text"
RATING_COLUMN = None  # chỉ dùng để audit, tuyệt đối không đưa vào model
EXCEL_SHEET_NAME = 0
MAX_ROWS = int(os.getenv("MIS495_MAX_ROWS", "0"))  # 0 = toàn bộ
EXPORT_XLSX = True
EXPECTED_INPUT_SHA256 = (
    '8dc7ea0a9b6999438c47f85eb210bfd6aa604662f8937f4b5d24b3daeef07c00'
)
OUTPUT_NAME = 'MIS495_Thy_Hau_Sentiment_Reproduced.xlsx'

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = BATCH_SIZE_GPU if DEVICE.type == "cuda" else BATCH_SIZE_CPU
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path(os.getenv("MIS495_OUTPUT_DIR", f"/content/mis495_reproducible_{RUN_ID}"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

config_view = {
    "model": MODEL_ID,
    "revision": MODEL_REVISION,
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "low_confidence_threshold": LOW_CONFIDENCE_THRESHOLD,
    "max_rows": MAX_ROWS or "all",
    "output_dir": str(OUTPUT_DIR),
    "output_name": OUTPUT_NAME,
    "expected_input_sha256": EXPECTED_INPUT_SHA256,
}
print(json.dumps(config_view, ensure_ascii=False, indent=2))
if IN_COLAB and DEVICE.type != "cuda":
    raise RuntimeError(
        "Chưa bật GPU. Chọn Runtime → Change runtime type → T4 GPU, "
        "sau đó Runtime → Run all lại từ đầu."
    )
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    print("PASS: GPU đã sẵn sàng.")


## Cell 3 — Upload đúng master CSV chính thức

In [ ]:
# Tải đúng một file CSV hoặc XLSX; khi test local có thể truyền MIS495_INPUT_PATH
LOCAL_INPUT_PATH = os.getenv("MIS495_INPUT_PATH", "").strip()

if LOCAL_INPUT_PATH:
    input_path = Path(LOCAL_INPUT_PATH)
    if not input_path.exists():
        raise FileNotFoundError(input_path)
    INPUT_NAME = input_path.name
    INPUT_BYTES = input_path.read_bytes()
elif IN_COLAB:
    print("Chọn đúng file reviews_master_text_v1.csv (47.466 review)...")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Cần tải đúng 1 file reviews_master_text_v1.csv.")
    INPUT_NAME, INPUT_BYTES = next(iter(uploaded.items()))
else:
    raise RuntimeError(
        "Không chạy trong Colab. Hãy đặt biến môi trường MIS495_INPUT_PATH khi test local."
    )

suffix = Path(INPUT_NAME).suffix.lower()
if suffix != ".csv":
    raise ValueError("Để tái lập workbook chính thức, input phải là file CSV.")

INPUT_SHA256 = hashlib.sha256(INPUT_BYTES).hexdigest()
print({
    "file": INPUT_NAME,
    "size_mb": round(len(INPUT_BYTES) / 1024 / 1024, 2),
    "sha256": INPUT_SHA256,
})
if INPUT_SHA256 != EXPECTED_INPUT_SHA256:
    raise ValueError(
        "Bạn đã upload nhầm file hoặc nhầm bản master. "
        "Hãy tải lại đúng file CSV được gửi kèm notebook. "
        f"Checksum cần có: {EXPECTED_INPUT_SHA256}; "
        f"checksum file vừa upload: {INPUT_SHA256}."
    )
print("PASS: checksum input chính thức khớp.")


## Cell 4 — Đọc dữ liệu và xem cấu trúc ban đầu

In [ ]:
# Đọc CSV robust theo encoding/delimiter; Excel dùng sheet đã cấu hình
def read_input_table(filename: str, payload: bytes) -> tuple[pd.DataFrame, str]:
    suffix = Path(filename).suffix.lower()
    if suffix in {".xlsx", ".xls"}:
        frame = pd.read_excel(io.BytesIO(payload), sheet_name=EXCEL_SHEET_NAME)
        return frame, f"excel_sheet={EXCEL_SHEET_NAME}"

    errors = []
    for encoding in ("utf-8-sig", "utf-8", "cp1258"):
        try:
            decoded = payload.decode(encoding)
            sample = decoded[:65536]
            try:
                delimiter = csv.Sniffer().sniff(
                    sample,
                    delimiters=",;\t|",
                ).delimiter
            except csv.Error:
                delimiter = ","  # an toàn cho CSV chỉ có một cột

            frame = pd.read_csv(io.StringIO(decoded), sep=delimiter)
            return frame, f"csv_encoding={encoding}; delimiter={delimiter!r}"
        except Exception as exc:
            errors.append(f"{encoding}: {exc}")
    raise ValueError("Không đọc được CSV: " + " | ".join(errors))


df_raw, READ_METHOD = read_input_table(INPUT_NAME, INPUT_BYTES)
INPUT_ROWS_TOTAL = len(df_raw)
if MAX_ROWS > 0:
    df_raw = df_raw.head(MAX_ROWS).copy()
if df_raw.empty:
    raise ValueError("File đầu vào không có dòng dữ liệu để phân tích.")

print({
    "read_method": READ_METHOD,
    "rows_total": INPUT_ROWS_TOTAL,
    "rows_processed": len(df_raw),
    "columns": len(df_raw.columns),
})
print("Danh sách cột:", list(df_raw.columns))
display(df_raw.head(5))


## Cell 5 — Chọn cột review và audit trước làm sạch

In [ ]:
# Tự tìm cột review và cột rating (rating chỉ dùng để audit)
TEXT_CANDIDATES = [
    "review_text", "review_text_raw", "review_content", "comment_text",
    "comment", "comments", "text", "content", "review", "feedback",
    "noi_dung", "nội_dung",
]
RATING_CANDIDATES = ["rating_star", "rating", "stars", "score", "star"]

lower_to_original = {str(column).lower(): column for column in df_raw.columns}
if TEXT_COLUMN is None:
    matches = [lower_to_original[name] for name in TEXT_CANDIDATES if name in lower_to_original]
    if not matches:
        raise ValueError(
            "Không tự tìm thấy cột review. Hãy sửa TEXT_COLUMN ở Cell cấu hình. "
            f"Các cột hiện có: {list(df_raw.columns)}"
        )
    TEXT_COLUMN = matches[0]
elif TEXT_COLUMN not in df_raw.columns:
    raise ValueError(f"Không có cột TEXT_COLUMN={TEXT_COLUMN!r}")

if RATING_COLUMN is None:
    rating_matches = [
        lower_to_original[name] for name in RATING_CANDIDATES if name in lower_to_original
    ]
    RATING_COLUMN = rating_matches[0] if rating_matches else None

ORIGINAL_COLUMNS = list(df_raw.columns)
RESERVED_OUTPUT_COLUMNS = {
    "_source_row", "sentiment_text_unicode", "sentiment_text_no_markup",
    "sentiment_text_clean", "sentiment_valid", "sentiment_skip_reason",
    "sentiment_character_count", "sentiment_duplicate_clean_text",
    "sentiment_token_length", "sentiment_truncated", "score_negative",
    "score_positive", "score_neutral", "sentiment_label",
    "sentiment_confidence", "sentiment_margin", "sentiment_score_signed",
    "low_confidence_flag", "sentiment_model", "sentiment_model_revision",
    "sentiment_score_sum",
}
reserved_collisions = sorted(
    str(column) for column in df_raw.columns if column in RESERVED_OUTPUT_COLUMNS
)
if reserved_collisions:
    raise ValueError(
        "Input đã có cột dành riêng cho output sentiment: "
        f"{reserved_collisions}. Hãy dùng file dữ liệu gốc."
    )

df_stage = df_raw.copy(deep=True)
df_stage.insert(0, "_source_row", np.arange(2, len(df_stage) + 2))
raw_text_as_string = df_stage[TEXT_COLUMN].fillna("").astype(str)

audit_before = pd.DataFrame([
    {"metric": "input_rows", "value": len(df_stage)},
    {"metric": "input_columns", "value": len(ORIGINAL_COLUMNS)},
    {"metric": "null_text", "value": int(df_stage[TEXT_COLUMN].isna().sum())},
    {"metric": "blank_text", "value": int(raw_text_as_string.str.strip().eq("").sum())},
    {"metric": "exact_duplicate_excess_before_cleaning", "value": int(df_raw.duplicated().sum())},
    {"metric": "duplicate_text_rows", "value": int(raw_text_as_string.duplicated(keep=False).sum())},
])

print({"text_column": TEXT_COLUMN, "rating_column_for_audit": RATING_COLUMN})
display(audit_before)
display(df_stage[["_source_row", TEXT_COLUMN]].head(10))

## Cell 6 — Phát hiện và loại header lặp

Hiển thị các dòng bị nhận diện là header trước khi loại.

In [ ]:
# Phát hiện header bị lặp thành dòng dữ liệu khi nhiều CSV được nối trực tiếp
header_match_count = pd.Series(0, index=df_stage.index, dtype="int64")
for column in ORIGINAL_COLUMNS:
    normalized_value = df_stage[column].fillna("").astype(str).str.strip().str.casefold()
    header_match_count += normalized_value.eq(str(column).strip().casefold()).astype("int64")

HEADER_MATCH_THRESHOLD = 1 if len(ORIGINAL_COLUMNS) == 1 else min(3, len(ORIGINAL_COLUMNS))
repeated_header_mask = header_match_count.ge(HEADER_MATCH_THRESHOLD)

excluded_header_rows = df_stage.loc[
    repeated_header_mask,
    ["_source_row", *ORIGINAL_COLUMNS],
].copy()
excluded_header_rows.insert(1, "cleaning_exclusion_reason", "repeated_header_row")

df_after_headers = df_stage.loc[~repeated_header_mask].copy()
REPEATED_HEADER_ROWS = len(excluded_header_rows)

print({
    "rows_before": len(df_stage),
    "repeated_header_rows_removed": REPEATED_HEADER_ROWS,
    "rows_after": len(df_after_headers),
    "header_match_threshold": HEADER_MATCH_THRESHOLD,
})
if REPEATED_HEADER_ROWS:
    display(excluded_header_rows.head(20))
else:
    print("Không phát hiện header lặp trong dữ liệu.")
display(df_after_headers[["_source_row", TEXT_COLUMN]].head(10))

## Cell 7 — Phát hiện và loại dòng trùng chính xác

Giữ lần xuất hiện đầu tiên và lưu toàn bộ dòng bị loại vào cleaning audit.

In [ ]:
# Loại dòng trùng chính xác trên toàn bộ cột gốc; giữ lần xuất hiện đầu tiên
exact_duplicate_mask = df_after_headers.duplicated(
    subset=ORIGINAL_COLUMNS,
    keep="first",
)
excluded_duplicate_rows = df_after_headers.loc[
    exact_duplicate_mask,
    ["_source_row", *ORIGINAL_COLUMNS],
].copy()
excluded_duplicate_rows.insert(1, "cleaning_exclusion_reason", "exact_duplicate_row")

df = (
    df_after_headers.loc[~exact_duplicate_mask]
    .copy()
    .reset_index(drop=True)
)
EXACT_DUPLICATE_ROWS = len(excluded_duplicate_rows)
SOURCE_ROW_SEQUENCE = df["_source_row"].tolist()

cleaning_exclusions = pd.concat(
    [excluded_header_rows, excluded_duplicate_rows],
    ignore_index=True,
)

print({
    "rows_before": len(df_after_headers),
    "exact_duplicate_rows_removed": EXACT_DUPLICATE_ROWS,
    "rows_after": len(df),
    "total_rows_removed": len(cleaning_exclusions),
})
if EXACT_DUPLICATE_ROWS:
    display(excluded_duplicate_rows[["_source_row", TEXT_COLUMN]].head(20))
else:
    print("Không có dòng trùng chính xác sau khi loại header lặp.")
display(df[["_source_row", TEXT_COLUMN]].head(10))

## Cell 8 — Chuẩn hóa Unicode và giải mã HTML entity

In [ ]:
# Bước làm sạch văn bản 1: null -> chuỗi rỗng, giải mã HTML entity, chuẩn hóa Unicode NFC
def normalize_unicode_and_entities(value: object) -> str:
    if pd.isna(value):
        return ""
    return unicodedata.normalize("NFC", html.unescape(str(value)))


df["sentiment_text_unicode"] = df[TEXT_COLUMN].map(normalize_unicode_and_entities)
unicode_changed = df[TEXT_COLUMN].fillna("").astype(str).ne(df["sentiment_text_unicode"])

print({
    "rows_processed": len(df),
    "rows_changed_by_unicode_or_html_entity": int(unicode_changed.sum()),
    "blank_after_step_1": int(df["sentiment_text_unicode"].str.strip().eq("").sum()),
})
display(
    df.loc[unicode_changed, ["_source_row", TEXT_COLUMN, "sentiment_text_unicode"]]
    .head(20)
)
display(df[["_source_row", TEXT_COLUMN, "sentiment_text_unicode"]].head(10))

## Cell 9 — Bỏ URL, thẻ HTML và ký tự điều khiển

In [ ]:
# Bước làm sạch văn bản 2: bỏ URL, thẻ HTML và ký tự điều khiển/zero-width
def remove_markup_urls_and_controls(value: str) -> str:
    text = re.sub(r"https?://\S+|www\.\S+", " ", value)
    text = re.sub(r"</?[A-Za-z][A-Za-z0-9]*(?:\s[^<>]*)?>", " ", text)
    return "".join(
        character
        for character in text
        if unicodedata.category(character) not in {"Cc", "Cf"}
        or character in {"\n", "\t", "\u200d"}
    )


df["sentiment_text_no_markup"] = df["sentiment_text_unicode"].map(
    remove_markup_urls_and_controls
)
markup_changed = df["sentiment_text_unicode"].ne(df["sentiment_text_no_markup"])

print({
    "rows_processed": len(df),
    "rows_changed_by_url_html_control_removal": int(markup_changed.sum()),
})
display(
    df.loc[
        markup_changed,
        ["_source_row", "sentiment_text_unicode", "sentiment_text_no_markup"],
    ].head(20)
)
display(df[["_source_row", "sentiment_text_unicode", "sentiment_text_no_markup"]].head(10))

## Cell 10 — Chuẩn hóa khoảng trắng và chốt text đầu vào model

In [ ]:
# Bước làm sạch văn bản 3: chuẩn hóa khoảng trắng và tạo cờ hợp lệ
def normalize_whitespace(value: str) -> str:
    return re.sub(r"\s+", " ", value).strip()


def clean_sentiment_text(value: object) -> str:
    step_1 = normalize_unicode_and_entities(value)
    step_2 = remove_markup_urls_and_controls(step_1)
    return normalize_whitespace(step_2)


cleaning_tests = {
    "negation_preserved": clean_sentiment_text("Không tốt") == "Không tốt",
    "emoji_preserved": "😡" in clean_sentiment_text("Quá tệ 😡"),
    "unicode_normalized": clean_sentiment_text("Sản phẩm") == "Sản phẩm",
    "blank_handled": clean_sentiment_text(None) == "",
    "url_removed": "http" not in clean_sentiment_text("ổn https://example.com"),
    "comparison_preserved": clean_sentiment_text("2 < 3 nhưng 5 > 4") == "2 < 3 nhưng 5 > 4",
    "html_removed": clean_sentiment_text("<b>tốt</b>") == "tốt",
    "whitespace_normalized": clean_sentiment_text("  giao\n hàng   nhanh ") == "giao hàng nhanh",
}
assert all(cleaning_tests.values()), cleaning_tests

df["sentiment_text_clean"] = df["sentiment_text_no_markup"].map(normalize_whitespace)
df["sentiment_valid"] = df["sentiment_text_clean"].str.len().gt(0)
df["sentiment_skip_reason"] = np.where(
    df["sentiment_valid"], "", "blank_or_null_text"
)
df["sentiment_character_count"] = df["sentiment_text_clean"].str.len()
df["sentiment_duplicate_clean_text"] = df["sentiment_text_clean"].duplicated(keep=False)

cleaning_flow = pd.DataFrame([
    {"stage": "Input", "rows": len(df_raw), "removed_at_stage": 0},
    {"stage": "After repeated-header removal", "rows": len(df_after_headers), "removed_at_stage": REPEATED_HEADER_ROWS},
    {"stage": "After exact-duplicate removal", "rows": len(df), "removed_at_stage": EXACT_DUPLICATE_ROWS},
    {"stage": "Valid text for inference", "rows": int(df["sentiment_valid"].sum()), "removed_at_stage": int((~df["sentiment_valid"]).sum())},
])

print("Cleaning unit tests:", cleaning_tests)
display(cleaning_flow)
display(
    df[
        [
            "_source_row", TEXT_COLUMN, "sentiment_text_unicode",
            "sentiment_text_no_markup", "sentiment_text_clean", "sentiment_valid",
        ]
    ].head(20)
)

## Cell 11 — Tạo tập text duy nhất để giảm thời gian inference

In [ ]:
# Chỉ inference mỗi clean text một lần; sau đó merge lại đủ mọi dòng gốc
unique_texts = (
    df.loc[df["sentiment_valid"], ["sentiment_text_clean"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
if unique_texts.empty:
    raise ValueError(
        "Không có review hợp lệ sau làm sạch. Hãy kiểm tra lại cột review và dữ liệu đầu vào."
    )

valid_rows = int(df["sentiment_valid"].sum())
saved_inferences = valid_rows - len(unique_texts)
print({
    "valid_rows": valid_rows,
    "unique_texts_to_infer": len(unique_texts),
    "duplicate_inferences_saved": saved_inferences,
})
display(unique_texts.head(10))


## Cell 12 — Tải tokenizer/model và xác minh mapping nhãn

In [ ]:
# Tải model tại revision cố định để có thể tái lập
load_started = time.time()
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=False,
)
model_config = AutoConfig.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=False,
)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=False,
).to(DEVICE)
model.eval()


def canonical_label(value: object) -> str:
    label = str(value).upper().strip()
    mapping = {
        "NEG": "Negative",
        "NEGATIVE": "Negative",
        "POS": "Positive",
        "POSITIVE": "Positive",
        "NEU": "Neutral",
        "NEUTRAL": "Neutral",
    }
    if label not in mapping:
        raise ValueError(f"Nhãn model không nhận diện được: {value!r}")
    return mapping[label]


ID_TO_LABEL = {
    int(index): canonical_label(label)
    for index, label in model_config.id2label.items()
}
EXPECTED_MAPPING = {0: "Negative", 1: "Positive", 2: "Neutral"}
assert ID_TO_LABEL == EXPECTED_MAPPING, {
    "expected": EXPECTED_MAPPING,
    "actual": ID_TO_LABEL,
}
assert model_config.num_labels == 3

print({
    "device": str(DEVICE),
    "model_id": MODEL_ID,
    "revision": MODEL_REVISION,
    "num_labels": model_config.num_labels,
    "id_to_label": ID_TO_LABEL,
    "model_max_length_config": getattr(model_config, "max_position_embeddings", None),
    "load_seconds": round(time.time() - load_started, 2),
})


## Cell 13 — Kiểm tra độ dài token và cảnh báo truncation

In [ ]:
# Tính độ dài token theo batch mà chưa cắt để biết review nào vượt MAX_LENGTH
def token_lengths(texts: list[str], batch_size: int = 512) -> list[int]:
    lengths = []
    for start in tqdm(range(0, len(texts), batch_size), desc="Token length audit"):
        batch = texts[start : start + batch_size]
        encoded = tokenizer(
            batch,
            add_special_tokens=True,
            truncation=False,
            padding=False,
        )
        lengths.extend(len(ids) for ids in encoded["input_ids"])
    return lengths


unique_texts["sentiment_token_length"] = token_lengths(
    unique_texts["sentiment_text_clean"].tolist()
)
unique_texts["sentiment_truncated"] = unique_texts["sentiment_token_length"].gt(MAX_LENGTH)

length_summary = unique_texts["sentiment_token_length"].describe(
    percentiles=[0.5, 0.9, 0.95, 0.99]
).to_frame("token_length")
print({
    "max_length_used": MAX_LENGTH,
    "texts_over_max_length": int(unique_texts["sentiment_truncated"].sum()),
    "truncation_rate": round(float(unique_texts["sentiment_truncated"].mean()), 4),
})
display(length_summary)
display(
    unique_texts.sort_values("sentiment_token_length", ascending=False).head(10)
)


## Cell 14 — Định nghĩa inference trả đủ ba score

Nhãn được lấy bằng `argmax` trên Negative, Positive và Neutral.

In [ ]:
# Batch inference với softmax 3 lớp
def predict_sentiment_scores(
    texts: list[str],
    batch_size: int = BATCH_SIZE,
) -> pd.DataFrame:
    all_probabilities = []
    started = time.time()

    for start in tqdm(range(0, len(texts), batch_size), desc="ViSoBERT inference"):
        batch = texts[start : start + batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
        with torch.inference_mode():
            logits = model(**encoded).logits
            probabilities = torch.softmax(logits, dim=-1)
        all_probabilities.append(probabilities.cpu().numpy())

    if not all_probabilities:
        return pd.DataFrame(
            columns=[
                "score_negative",
                "score_positive",
                "score_neutral",
                "sentiment_label",
                "sentiment_confidence",
                "sentiment_margin",
                "sentiment_score_signed",
                "low_confidence_flag",
            ]
        )

    probs = np.vstack(all_probabilities).astype("float32")
    assert probs.shape == (len(texts), 3), probs.shape

    label_indices = probs.argmax(axis=1)
    sorted_probs = np.sort(probs, axis=1)
    confidence = sorted_probs[:, -1]
    margin = sorted_probs[:, -1] - sorted_probs[:, -2]

    result = pd.DataFrame({
        "score_negative": probs[:, 0],
        "score_positive": probs[:, 1],
        "score_neutral": probs[:, 2],
        "sentiment_label": [ID_TO_LABEL[int(index)] for index in label_indices],
        "sentiment_confidence": confidence,
        "sentiment_margin": margin,
        "sentiment_score_signed": probs[:, 1] - probs[:, 0],
        "low_confidence_flag": confidence < LOW_CONFIDENCE_THRESHOLD,
    })
    result.attrs["elapsed_seconds"] = time.time() - started
    return result


print(
    "Quy tắc: sentiment_label = argmax(score_negative, score_positive, score_neutral)."
)
print(
    "sentiment_score_signed = score_positive - score_negative; Neutral vẫn dùng score_neutral riêng."
)


## Cell 15 — Smoke test: nhìn score và nhãn trên ví dụ

In [ ]:
# Ba nhóm ví dụ; chỉ kiểm tra shape/tổng score, không giả định model luôn đúng ngữ cảnh
smoke_texts = [
    "Sản phẩm rất tốt, đóng gói kỹ và giao hàng nhanh 👍",
    "Hàng quá tệ, bị vỡ và giao chậm, rất thất vọng 😡",
    "Đã nhận hàng, chưa sử dụng nên chưa đánh giá được.",
    "Giá hơi cao nhưng chất lượng tốt.",
    "Hàng không bị bể vỡ, shop tư vấn nhiệt tình.",
    "Giao hàng bình thường, sản phẩm như mô tả.",
]
smoke_scores = predict_sentiment_scores(smoke_texts, batch_size=min(BATCH_SIZE, 6))
smoke_result = pd.concat(
    [pd.DataFrame({"review_text": smoke_texts}), smoke_scores],
    axis=1,
)
assert np.allclose(
    smoke_result[["score_negative", "score_positive", "score_neutral"]].sum(axis=1),
    1.0,
    atol=1e-5,
)
display(smoke_result)


## Cell 16 — Chạy sentiment trên toàn bộ text duy nhất

In [ ]:
# Inference toàn bộ unique clean text rồi ghép score vào bảng unique
inference_started = time.time()
unique_scores = predict_sentiment_scores(
    unique_texts["sentiment_text_clean"].tolist(),
    batch_size=BATCH_SIZE,
)
INFERENCE_SECONDS = time.time() - inference_started
unique_scored = pd.concat([unique_texts, unique_scores], axis=1)
unique_scored["sentiment_model"] = MODEL_ID
unique_scored["sentiment_model_revision"] = MODEL_REVISION

print({
    "unique_texts_scored": len(unique_scored),
    "elapsed_seconds": round(INFERENCE_SECONDS, 2),
    "rows_per_second": round(len(unique_scored) / max(INFERENCE_SECONDS, 1e-9), 2),
})
display(
    unique_scored[
        [
            "sentiment_text_clean",
            "score_negative",
            "score_positive",
            "score_neutral",
            "sentiment_label",
            "sentiment_confidence",
            "sentiment_score_signed",
        ]
    ].head(20)
)


## Cell 17 — Ghép score về từng dòng review đã làm sạch

In [ ]:
# Merge score về các dòng đã qua cleaning; giữ nguyên lineage _source_row
RESULT_COLUMNS = [
    "sentiment_text_clean",
    "sentiment_token_length",
    "sentiment_truncated",
    "score_negative",
    "score_positive",
    "score_neutral",
    "sentiment_label",
    "sentiment_confidence",
    "sentiment_margin",
    "sentiment_score_signed",
    "low_confidence_flag",
    "sentiment_model",
    "sentiment_model_revision",
]

df_result = df.merge(
    unique_scored[RESULT_COLUMNS],
    on="sentiment_text_clean",
    how="left",
    validate="many_to_one",
)

invalid_mask = ~df_result["sentiment_valid"]
df_result.loc[invalid_mask, "sentiment_label"] = "Unclassified"
df_result.loc[invalid_mask, "sentiment_model"] = MODEL_ID
df_result.loc[invalid_mask, "sentiment_model_revision"] = MODEL_REVISION
df_result["sentiment_score_sum"] = df_result[
    ["score_negative", "score_positive", "score_neutral"]
].sum(axis=1, min_count=1)

assert len(df_result) == len(df)
assert df_result["_source_row"].is_unique

print({
    "input_rows_before_cleaning": len(df_raw),
    "rows_after_cleaning": len(df),
    "output_rows": len(df_result),
    "scored_rows": int(df_result["sentiment_valid"].sum()),
    "unclassified_rows": int(df_result["sentiment_label"].eq("Unclassified").sum()),
})
display(
    df_result[
        [
            "_source_row",
            TEXT_COLUMN,
            "score_negative",
            "score_positive",
            "score_neutral",
            "sentiment_label",
            "sentiment_confidence",
            "sentiment_score_signed",
        ]
    ].head(20)
)

## Cell 18 — Tổng hợp kết quả, biểu đồ và ca confidence thấp

In [ ]:
# Bảng phân bố nhãn và mức tin cậy
label_summary = (
    df_result.groupby("sentiment_label", dropna=False, as_index=False)
    .agg(
        review_count=("_source_row", "size"),
        mean_confidence=("sentiment_confidence", "mean"),
        median_signed_score=("sentiment_score_signed", "median"),
        low_confidence_count=("low_confidence_flag", "sum"),
    )
)
label_summary["review_share"] = label_summary["review_count"] / len(df_result)
display(label_summary.sort_values("review_count", ascending=False))

valid_result = df_result.loc[df_result["sentiment_valid"]].copy()
confidence_summary = valid_result["sentiment_confidence"].describe(
    percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95]
).to_frame("sentiment_confidence")
display(confidence_summary)

low_confidence_examples = valid_result.sort_values(
    ["sentiment_confidence", "sentiment_margin"],
    ascending=[True, True],
).head(20)
display(
    low_confidence_examples[
        [
            TEXT_COLUMN,
            "score_negative",
            "score_positive",
            "score_neutral",
            "sentiment_label",
            "sentiment_confidence",
            "sentiment_margin",
        ]
    ]
)

if RATING_COLUMN and RATING_COLUMN in df_result.columns:
    print("Đối chiếu rating chỉ để audit; rating KHÔNG được dùng làm input model.")
    display(pd.crosstab(df_result[RATING_COLUMN], df_result["sentiment_label"], margins=True))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
plot_labels = label_summary.loc[label_summary["sentiment_label"] != "Unclassified"]
sentiment_colors = {
    "Positive": "#2F6B5F",
    "Negative": "#C44E52",
    "Neutral": "#DDC36A",
}
plot_colors = plot_labels["sentiment_label"].map(sentiment_colors).fillna("#808080")
axes[0].bar(plot_labels["sentiment_label"], plot_labels["review_count"], color=plot_colors)
axes[0].set_title("Số review theo sentiment")
axes[0].set_ylabel("Số review")
axes[1].hist(valid_result["sentiment_confidence"], bins=20, color="#4C72B0", edgecolor="white")
axes[1].axvline(LOW_CONFIDENCE_THRESHOLD, color="#C44E52", linestyle="--", label="low-confidence threshold")
axes[1].set_title("Phân bố confidence")
axes[1].set_xlabel("Sentiment confidence")
axes[1].legend()
plt.tight_layout()
CHART_PATH = OUTPUT_DIR / "sentiment_distribution.png"
fig.savefig(CHART_PATH, dpi=160, bbox_inches="tight")
plt.show()


## Cell 19 — QA cleaning, score, argmax và lineage

In [ ]:
# QA cho cleaning, score, nhãn, merge và lineage
score_columns = ["score_negative", "score_positive", "score_neutral"]
valid_mask = df_result["sentiment_valid"]
valid_scores = df_result.loc[valid_mask, score_columns]
invalid_scores = df_result.loc[~valid_mask, score_columns]
valid_score_array = valid_scores.to_numpy(dtype="float64")
score_column_to_label = {
    "score_negative": "Negative",
    "score_positive": "Positive",
    "score_neutral": "Neutral",
}
expected_labels_from_scores = valid_scores.idxmax(axis=1).map(score_column_to_label)
sorted_valid_scores = np.sort(valid_score_array, axis=1)
expected_margin = sorted_valid_scores[:, -1] - sorted_valid_scores[:, -2]
expected_low_confidence = (
    df_result.loc[valid_mask, "sentiment_confidence"] < LOW_CONFIDENCE_THRESHOLD
)

duplicate_prediction_consistent = True
if df_result.loc[valid_mask, "sentiment_duplicate_clean_text"].any():
    duplicate_groups = df_result.loc[
        valid_mask & df_result["sentiment_duplicate_clean_text"]
    ].groupby("sentiment_text_clean")[score_columns]
    duplicate_prediction_consistent = bool(
        (duplicate_groups.nunique(dropna=False).max(axis=1) <= 1).all()
    )

qa_items = [
    ("cleaning_row_reconciliation", len(df_raw) == len(df_result) + len(cleaning_exclusions), f"{len(df_raw)}={len(df_result)}+{len(cleaning_exclusions)}"),
    ("repeated_headers_removed", REPEATED_HEADER_ROWS == len(excluded_header_rows), REPEATED_HEADER_ROWS),
    ("exact_duplicates_removed", not df.duplicated(subset=ORIGINAL_COLUMNS).any(), EXACT_DUPLICATE_ROWS),
    ("source_row_unique", df_result["_source_row"].is_unique, int(df_result["_source_row"].duplicated().sum())),
    ("retained_source_order_preserved", df_result["_source_row"].tolist() == SOURCE_ROW_SEQUENCE, "exact retained sequence"),
    ("all_original_columns_present", all(column in df_result.columns for column in ORIGINAL_COLUMNS), ORIGINAL_COLUMNS),
    ("retained_original_values_unchanged", df_result[ORIGINAL_COLUMNS].equals(df[ORIGINAL_COLUMNS]), "retained input columns"),
    ("raw_text_unchanged", df_result[TEXT_COLUMN].equals(df[TEXT_COLUMN]), TEXT_COLUMN),
    ("cleaning_unit_tests", all(cleaning_tests.values()), cleaning_tests),
    ("model_label_mapping", ID_TO_LABEL == EXPECTED_MAPPING, ID_TO_LABEL),
    ("valid_scores_present", not valid_scores.isna().any().any(), int(valid_scores.isna().sum().sum())),
    ("valid_scores_finite", bool(np.isfinite(valid_score_array).all()), valid_scores.shape),
    ("valid_scores_in_0_1", bool(((valid_scores >= 0) & (valid_scores <= 1)).all().all()), "[0,1]"),
    ("valid_scores_sum_to_1", bool(np.allclose(valid_scores.sum(axis=1), 1.0, atol=1e-5)), float((valid_scores.sum(axis=1) - 1.0).abs().max())),
    ("invalid_scores_are_nan", bool(invalid_scores.isna().all().all()), int(invalid_scores.notna().sum().sum())),
    ("invalid_label_unclassified", bool(df_result.loc[~valid_mask, "sentiment_label"].eq("Unclassified").all()), int((~valid_mask).sum())),
    ("valid_label_set", set(df_result.loc[valid_mask, "sentiment_label"].unique()).issubset({"Positive", "Negative", "Neutral"}), sorted(df_result.loc[valid_mask, "sentiment_label"].unique())),
    ("label_equals_score_argmax", df_result.loc[valid_mask, "sentiment_label"].equals(expected_labels_from_scores), "argmax(NEG,POS,NEU)"),
    ("signed_score_formula", bool(np.allclose(df_result.loc[valid_mask, "sentiment_score_signed"], df_result.loc[valid_mask, "score_positive"] - df_result.loc[valid_mask, "score_negative"], atol=1e-7)), "POS-NEG"),
    ("confidence_formula", bool(np.allclose(df_result.loc[valid_mask, "sentiment_confidence"], valid_scores.max(axis=1), atol=1e-7)), "max(scores)"),
    ("margin_formula", bool(np.allclose(df_result.loc[valid_mask, "sentiment_margin"], expected_margin, atol=1e-7)), "top1-top2"),
    ("low_confidence_formula", df_result.loc[valid_mask, "low_confidence_flag"].astype(bool).equals(expected_low_confidence), f"confidence<{LOW_CONFIDENCE_THRESHOLD}"),
    ("margin_nonnegative", bool(df_result.loc[valid_mask, "sentiment_margin"].ge(0).all()), float(df_result.loc[valid_mask, "sentiment_margin"].min())),
    ("duplicate_predictions_consistent", duplicate_prediction_consistent, int(df_result["sentiment_duplicate_clean_text"].sum())),
]

qa_report = pd.DataFrame([
    {"check": name, "passed": bool(passed), "observed": str(observed)}
    for name, passed, observed in qa_items
])
display(qa_report)
if not qa_report["passed"].all():
    failed = qa_report.loc[~qa_report["passed"]]
    raise AssertionError("QA thất bại:\n" + failed.to_string(index=False))
print(f"PASS: {int(qa_report['passed'].sum())}/{len(qa_report)} QA checks")

## Cell 20 — Tạo Error Audit 50 và Validation 100 unique blinded

In [ ]:
# Error Audit 50: mỗi clean text chỉ xuất hiện một lần
def build_error_audit_sample(frame: pd.DataFrame, target: int = 50) -> pd.DataFrame:
    pool = frame.loc[frame["sentiment_valid"]].drop_duplicates("sentiment_text_clean").copy()
    quotas = {"Positive": 17, "Negative": 17, "Neutral": 16}
    selected_indices = []

    for offset, (label, quota) in enumerate(quotas.items()):
        group = pool.loc[pool["sentiment_label"].eq(label)].sort_values(
            ["sentiment_confidence", "sentiment_margin"], ascending=[True, True]
        )
        low_n = min(len(group), math.ceil(quota / 2))
        low = group.head(low_n)
        selected_indices.extend(low.index.tolist())
        remaining = group.drop(index=low.index)
        random_n = min(quota - low_n, len(remaining))
        if random_n:
            selected_indices.extend(
                remaining.sample(n=random_n, random_state=RANDOM_SEED + offset).index.tolist()
            )

    selected_indices = list(dict.fromkeys(selected_indices))
    if len(selected_indices) < target:
        remaining = pool.drop(index=selected_indices)
        fill_n = min(target - len(selected_indices), len(remaining))
        if fill_n:
            selected_indices.extend(
                remaining.sample(n=fill_n, random_state=RANDOM_SEED + 999).index.tolist()
            )

    sample = pool.loc[selected_indices[:target]].copy()
    sample["human_sentiment_label"] = ""
    sample["human_note"] = ""
    return sample[[
        "_source_row", TEXT_COLUMN, "score_negative", "score_positive",
        "score_neutral", "sentiment_label", "sentiment_confidence",
        "sentiment_margin", "human_sentiment_label", "human_note",
    ]].reset_index(drop=True)


error_audit_sample_50 = build_error_audit_sample(df_result, target=50)

# Random Validation 100: random trên clean text unique và ẩn toàn bộ dự đoán
validation_pool = (
    df_result.loc[df_result["sentiment_valid"], ["_source_row", TEXT_COLUMN, "sentiment_text_clean"]]
    .drop_duplicates("sentiment_text_clean")
    .copy()
)
random_validation_target = min(100, len(validation_pool))
random_validation_100 = (
    validation_pool.sample(n=random_validation_target, random_state=RANDOM_SEED)
    .sort_values("_source_row")
    .reset_index(drop=True)
)
assert not random_validation_100["sentiment_text_clean"].duplicated().any()
random_validation_100 = random_validation_100[["_source_row", TEXT_COLUMN]]
random_validation_100["human_sentiment_label"] = ""
random_validation_100["human_note"] = ""

print({
    "error_audit_rows": len(error_audit_sample_50),
    "error_audit_predicted_labels": error_audit_sample_50["sentiment_label"].value_counts().to_dict(),
    "random_blinded_validation_unique_texts": len(random_validation_100),
})
print("Error Audit 50 — có score và nhãn model để tìm lỗi:")
display(error_audit_sample_50.head(20))
print("Random Validation 100 — blinded, 100 text unique:")
display(random_validation_100.head(20))

## Phần A — Aspect-based sentiment V1B

In [ ]:
# Chuẩn bị bảng chung cho ABSA và topic modeling
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer

RULE_VERSION = "ABSA_RULE_V1B_UNVALIDATED"
ASPECT_SLUGS = {
    "Giao hàng": "delivery",
    "Đóng gói": "packaging",
    "Chất lượng & hiệu quả": "quality",
    "Hàng thật & hàng giả": "authenticity",
    "Tư vấn & dịch vụ": "service",
    "Giá": "price",
}

reviews = df_result.copy()
if "review_text" not in reviews.columns:
    reviews["review_text"] = reviews[TEXT_COLUMN]

if "review_id_v2" not in reviews.columns:
    hash_columns = [
        column for column in
        ["shop_name", "item_id", "reviewer_username", "review_date", "review_text", "seller_reply"]
        if column in reviews.columns
    ]
    reviews["review_id_v2"] = (
        reviews[hash_columns]
        .fillna("")
        .astype(str)
        .agg("\x1f".join, axis=1)
        .map(lambda value: hashlib.sha256(value.encode("utf-8")).hexdigest()[:24])
    )

if reviews["review_id_v2"].duplicated().any():
    raise AssertionError("review_id_v2 phải duy nhất sau bước làm sạch.")

reviews["review_date_parsed"] = pd.to_datetime(
    reviews.get("review_date", pd.Series(pd.NaT, index=reviews.index)),
    errors="coerce",
)
print({
    "reviews_for_analysis": len(reviews),
    "shops": sorted(reviews["shop_name"].dropna().astype(str).unique().tolist()),
    "unique_review_id": int(reviews["review_id_v2"].nunique()),
})


In [ ]:
# 5) Rule ABSA V1B — trích nguyên bản từ scripts/mis495_build_v1.py
ASPECTS = {
    "Giao hàng": {
        "mention": (
            r"\b(?:giao hàng|giao hang|vận chuyển|van chuyen|nhận hàng|nhan hang|"
            r"đơn hàng|don hang|hỏa tốc|hoa toc|shipper|ship hàng|ship hang)\b"
            r"|\bgiao (?:nhanh|chậm|cham|lâu|lau|trễ|tre|đúng|dung|sai|"
            r"thiếu|thieu|nhầm|nham)\b"
        ),
        "positive": (
            r"\bgiao(?: hàng| hang)? (?:rất |rat )?nhanh\b|"
            r"\bgiao (?:đúng|dung)(?: hẹn| hen| giờ| gio| ngày| ngay)?\b|"
            r"\bnhận(?: hàng)? sớm\b|\bnhan(?: hang)? som\b|"
            r"\bvận chuyển nhanh\b|\bvan chuyen nhanh\b|"
            r"\bshipper .{0,30}(?:nhiệt tình|nhiet tinh|thân thiện|than thien|lịch sự|lich su)\b"
        ),
        "negative": (
            r"\bgiao(?: hàng| hang)? (?:quá |qua )?(?:chậm|cham|trễ|tre)\b|"
            r"\bgiao(?: hàng| hang)? (?:quá |qua )?(?:lâu(?! dài)|lau(?! dai))\b|"
            r"\bvận chuyển (?:chậm|trễ|quá lâu)\b|\bvan chuyen (?:cham|tre|qua lau)\b|"
            r"\b(?:không|khong|ko) giao\b|\bgiao (?:sai|thiếu|thieu|nhầm|nham)\b|"
            r"\b(?:thất lạc|that lac)\b|\bgiao .{0,12}(?:không|khong|ko) nhanh\b|"
            r"\bshipper .{0,30}(?:tệ|te|khó chịu|kho chiu|bất lịch sự|bat lich su)\b"
        ),
        "negative_blockers": [
            r"\b(?:không|khong|ko) nhanh (?:cũng |cung )?"
            r"(?:không|khong|ko) (?:chậm|cham)\b"
        ],
        "neutral_override": (
            r"\b(?:không|khong|ko) nhanh (?:cũng |cung )?"
            r"(?:không|khong|ko) (?:chậm|cham)\b"
        ),
    },
    "Đóng gói": {
        "mention": (
            r"\b(?:đóng gói|dong goi|gói hàng|goi hang|đóng hàng|dong hang|"
            r"bao bì|bao bi|bọc hàng|boc hang|niêm phong|niem phong)\b|"
            r"\b(?:chai|lọ|lo|hộp|hop|bao|túi|tui|gói|goi|bịch|bich|"
            r"thùng|thung|hàng|hang) .{0,12}"
            r"(?:bể|be(?! bé| be)|vỡ|vo|móp|mop|rách|rach|rò rỉ|ro ri|ướt|uot)\b|"
            r"\b(?:bung nắp|bung nap|chảy ra|chay ra|tràn ra|tran ra)\b"
        ),
        "positive": (
            r"\b(?:đóng gói|dong goi|gói hàng|goi hang) .{0,25}"
            r"(?:cẩn thận|can than|kỹ|ky|tốt|tot|chắc chắn|chac chan|đẹp|dep)\b|"
            r"\b(?:bao bì|bao bi) .{0,15}(?:đẹp|dep|chắc chắn|chac chan)\b|"
            r"\b(?:không|khong|ko|k|chẳng|chả|chưa) (?:bị |bi )?"
            r"(?:bể|be(?! bé| be)|vỡ|vo|móp|mop|rách|rach|rò rỉ|ro ri)"
            r"(?:\s*(?:/|,|và|va|hay)?\s*"
            r"(?:bể|be(?! bé| be)|vỡ|vo|móp|mop|rách|rach|rò rỉ|ro ri))*\b"
        ),
        "negative": (
            r"\b(?:đóng gói|dong goi|đóng hàng|dong hang|gói hàng|goi hang) .{0,25}"
            r"(?:sơ sài|so sai|quá tệ|qua te|kém|kem|không kỹ|khong ky|"
            r"không cẩn thận|khong can than|không chắc chắn|khong chac chan)\b|"
            r"\b(?:chai|lọ|lo|hộp|hop|bao|túi|tui|gói|goi|bịch|bich|"
            r"thùng|thung|hàng|hang) .{0,12}"
            r"(?:bể|be(?! bé| be)|vỡ|vo|móp|mop|rách|rach|rò rỉ|ro ri|ướt|uot)\b|"
            r"\b(?:bung nắp|bung nap|chảy ra|chay ra|tràn ra|tran ra)\b"
        ),
        "negative_blockers": [
            r"\b(?:không|khong|ko|k|chẳng|chả|chưa) (?:bị |bi )?"
            r"(?:bể|be(?! bé| be)|vỡ|vo|móp|mop|rách|rach|rò rỉ|ro ri)"
            r"(?:\s*(?:/|,|và|va|hay)?\s*"
            r"(?:bể|be(?! bé| be)|vỡ|vo|móp|mop|rách|rach|rò rỉ|ro ri))*\b"
        ],
    },
    "Chất lượng & hiệu quả": {
        "mention": (
            r"\b(?:chất lượng|chat luong|hiệu quả|hieu qua|tác dụng|tac dung|"
            r"công dụng|cong dung|nảy mầm|nay mam|sinh trưởng|sinh truong|"
            r"phát triển|phat trien|đúng mô tả|dung mo ta|sai mô tả|sai mo ta)\b|"
            r"\b(?:lâu héo|lau heo|hạn chế rụng|han che rung)\b|"
            r"\b(?:sản phẩm|san pham|hàng|hang) .{0,15}"
            r"\b(?:tốt|tot|ổn|on|kém|kem|hỏng|hong|lỗi|loi)\b|"
            r"\b(?:cây|cay) .{0,15}\b"
            r"(?:chết|chet|héo|heo|cháy lá|chay la|khỏe|khoe)\b|"
            r"\b(?:kiến|kien|sâu|sau|rệp|rep|bọ|bo) .{0,12}\b(?:chết|chet)\b"
        ),
        "positive": (
            r"\b(?:chất lượng|chat luong) .{0,20}\b"
            r"(?:tốt|tot|ổn|on|tuyệt vời|tuyet voi)\b|"
            r"\b(?:rất|rat|có|co) (?:hiệu quả|hieu qua)\b|"
            r"\b(?:hiệu quả|hieu qua) (?:cao|tốt|tot|rõ rệt|ro ret)\b|"
            r"\b(?:tác dụng|tac dung) (?:tốt|tot|nhanh)\b|"
            r"\b(?:nảy mầm|nay mam) (?:tốt|tot|đều|deu|nhanh|cao)\b|"
            r"\b(?:cây|cay) .{0,36}\b"
            r"(?:phát triển (?:rất |rat )?tốt|phat trien (?:rat )?tot|"
            r"khỏe|khoe|xanh tốt|xanh tot)\b|"
            r"\b(?:kiến|kien|sâu|sau|rệp|rep|bọ|bo) .{0,12}\b(?:chết|chet)\b|"
            r"\b(?:lâu héo|lau heo|hạn chế rụng|han che rung)\b|"
            r"\b(?:cây|cay) .{0,12}(?:không|khong|ko|chẳng|chả|chưa)"
            r"(?: bị| bi)? .{0,4}(?:chết|chet|héo|heo|cháy lá|chay la)\b|"
            r"\b(?:đúng mô tả|dung mo ta)\b|"
            r"\b(?:dùng|dung|sử dụng|su dung|sản phẩm|san pham|hàng|hang) "
            r"(?:sử dụng |su dung )?(?:rất |rat )?"
            r"(?:tốt|tot|ok|ổn|on)\b|"
            r"\b(?:hàng|hang|sản phẩm|san pham) .{0,8}"
            r"(?:không|khong|ko|k|chẳng|chả|chưa)(?: bị| bi)? .{0,4}\b"
            r"(?:kém|kem|hư|hu|hỏng|hong|lỗi|loi)\b|"
            r"\b(?:không|khong|ko) (?:lo|sợ|so) "
            r"(?:hàng|hang|sản phẩm|san pham) .{0,8}\b"
            r"(?:kém|kem|hư|hu|hỏng|hong|lỗi|loi)\b"
        ),
        "negative": (
            r"\b(?:chất lượng|chat luong) .{0,20}\b"
            r"(?:kém|kem|tệ|te|xấu|xau|không tốt|khong tot)\b|"
            r"\b(?:không|khong|ko|chưa thấy|chua thay) (?:có |co )?"
            r"(?:hiệu quả|hieu qua|tác dụng|tac dung)\b|"
            r"\b(?:không nảy|khong nay)(?: mầm| mam)?\b|"
            r"\b(?:nảy mầm|nay mam) (?:kém|kem|thấp|thap)\b|"
            r"\b(?:sai|không đúng|khong dung) (?:mô tả|mo ta)\b|"
            r"\b(?:hàng|hang|sản phẩm|san pham) .{0,12}\b"
            r"(?:kém|kem|hư|hu|hỏng|hong|lỗi|loi)\b|"
            r"\b(?:kiến|kien|sâu|sau|rệp|rep|bọ|bo) .{0,8}\b"
            r"(?:không|khong|ko|chẳng|chả|chưa)(?: bị| bi)? .{0,4}"
            r"(?:chết|chet)\b|"
            r"\b(?:cây|cay) .{0,12}\b(?:chết|chet|héo|heo|cháy lá|chay la)\b"
        ),
        "negative_blockers": [
            r"\b(?:lâu héo|lau heo|hạn chế rụng|han che rung)\b",
            r"\b(?:cây|cay) .{0,12}(?:không|khong|ko|chẳng|chả|chưa)"
            r"(?: bị| bi)? .{0,4}(?:chết|chet|héo|heo|cháy lá|chay la)\b",
            r"\b(?:không|khong|ko|chẳng|chả|chưa)(?: có| co)? "
            r"(?:hàng|hang|sản phẩm|san pham) .{0,8}\b"
            r"(?:kém|kem|hư|hu|hỏng|hong|lỗi|loi)\b",
            r"\b(?:hàng|hang|sản phẩm|san pham) .{0,8}"
            r"(?:không|khong|ko|k|chẳng|chả|chưa)(?: bị| bi)? .{0,4}\b"
            r"(?:kém|kem|hư|hu|hỏng|hong|lỗi|loi)\b",
            r"\b(?:không|khong|ko) (?:lo|sợ|so) "
            r"(?:hàng|hang|sản phẩm|san pham) .{0,8}\b"
            r"(?:kém|kem|hư|hu|hỏng|hong|lỗi|loi)\b",
            r"\b(?:sợ|so|lo|ngại|ngai) (?:gặp |gap )?"
            r"(?:hàng|hang|sản phẩm|san pham) .{0,8}\b"
            r"(?:kém|kem|hư|hu|hỏng|hong|lỗi|loi)\b",
            r"\b(?:hàng|hang|sản phẩm|san pham) .{0,8}\b"
            r"(?:kém|kem|hư|hu|hỏng|hong|lỗi|loi) (?:bao bì|bao bi)\b",
        ],
        "neutral_override": (
            r"\b(?:chưa dùng|chua dung|chưa xài|chua xai|chưa thử|chua thu|"
            r"chưa biết|chua biet|mới nhận|moi nhan)\b"
        ),
    },
    "Hàng thật & hàng giả": {
        "mention": (
            r"\b(?:chính hãng|chinh hang|hàng thật|hang that|hàng giả|hang gia|"
            r"hàng nhái|hang nhai|fake|auth|tem chống giả|tem chong gia|"
            r"nguồn gốc|nguon goc|xuất xứ|xuat xu|nhãn mác|nhan mac|"
            r"hàng chuẩn|hang chuan|mã vạch|ma vach|qr)\b"
        ),
        "positive": (
            r"\b(?:chính hãng|chinh hang|hàng thật|hang that|hàng chuẩn|hang chuan|"
            r"tem chống giả|tem chong gia|nguồn gốc rõ ràng|nguon goc ro rang|"
            r"không sợ hàng giả|khong so hang gia)\b"
        ),
        "negative": (
            r"\b(?:hàng giả|hang gia|hàng nhái|hang nhai|fake|không chính hãng|"
            r"khong chinh hang|tem giả|tem gia|nghi hàng giả|nghi hang gia|"
            r"không rõ nguồn gốc|khong ro nguon goc|nhãn mác mờ|nhan mac mo)\b"
        ),
        "negative_blockers": [
            r"\b(?:tem chống giả|tem chong gia|không sợ hàng giả|khong so hang gia|"
            r"tránh hàng giả|tranh hang gia|tránh hàng fake|tranh hang fake)\b"
        ],
    },
    "Tư vấn & dịch vụ": {
        "mention": (
            r"\b(?:tư vấn|tu van|phản hồi|phan hoi|trả lời|tra loi|chăm sóc|cham soc|"
            r"hỗ trợ|ho tro|thái độ|thai do|inbox|nhắn tin|nhan tin|giải đáp|giai dap)\b|"
            r"\bshop .{0,25}(?:nhiệt tình|nhiet tinh|chu đáo|chu dao)\b"
        ),
        "positive": (
            r"\b(?:tư vấn|tu van|phản hồi|phan hoi|trả lời|tra loi|hỗ trợ|ho tro|"
            r"giải đáp|giai dap) "
            r"(?:(?!(?:nhưng|nhung|mà|ma|tuy nhiên|tuy nhien|"
            r"giao(?: hàng)?|giao(?: hang)?|vận chuyển|van chuyen|"
            r"đóng gói|dong goi|chất lượng|chat luong|giá|gia))"
            r"[^.!?;\n]){0,25}\b"
            r"(?:nhanh|tốt|tot|nhiệt tình|nhiet tinh|chu đáo|chu dao)\b|"
            r"\bshop (?:(?!(?:nhưng|nhung|mà|ma|tuy nhiên|tuy nhien|"
            r"giao(?: hàng)?|giao(?: hang)?|vận chuyển|van chuyen|"
            r"đóng gói|dong goi|chất lượng|chat luong|giá|gia))"
            r"[^.!?;\n]){0,25}\b(?:nhiệt tình|nhiet tinh|chu đáo|chu dao|"
            r"hỗ trợ tốt|ho tro tot|tư vấn tốt|tu van tot)\b"
        ),
        "negative": (
            r"\b(?:không|khong|ko) (?:tư vấn|tu van|phản hồi|phan hoi|trả lời|"
            r"tra loi|hỗ trợ|ho tro|giải đáp|giai dap)\b|"
            r"\b(?:tư vấn|tu van|phản hồi|phan hoi|trả lời|tra loi|hỗ trợ|ho tro|"
            r"giải đáp|giai dap) "
            r"(?:(?!(?:nhưng|nhung|mà|ma|tuy nhiên|tuy nhien|"
            r"giao(?: hàng)?|giao(?: hang)?|vận chuyển|van chuyen|"
            r"đóng gói|dong goi|chất lượng|chat luong|giá|gia))"
            r"[^.!?;\n]){0,25}\b(?:chậm|cham|kém|kem|tệ|te)\b|"
            r"\b(?:thái độ|thai do) .{0,20}(?:tệ|te|kém|kem|khó chịu|kho chiu)\b"
        ),
    },
    "Giá": {
        "mention": (
            r"\b(?:giá cả|gia ca|giá tiền|gia tien|giá thành|gia thanh|giá bán|gia ban|"
            r"giá rẻ|gia re|giá tốt|gia tot|giá cao|gia cao|giá đắt|gia dat|"
            r"giá quá cao|gia qua cao|giá hơi cao|gia hoi cao|"
            r"giá khá cao|gia kha cao|giá hơi đắt|gia hoi dat|"
            r"giá khá đắt|gia kha dat|"
            r"giá mềm|gia mem|đáng tiền|dang tien|hợp lý|hop ly|"
            r"phí ship|phi ship|phí vận chuyển|phi van chuyen|khuyến mãi|khuyen mai|voucher)\b"
        ),
        "positive": (
            r"\b(?:giá rẻ|gia re|giá tốt|gia tot|giá hợp lý|gia hop ly|"
            r"đáng tiền|dang tien|hợp lý|hop ly|giá mềm|gia mem)\b"
        ),
        "negative": (
            r"\b(?:giá đắt|gia dat|giá cao|gia cao|không đáng tiền|khong dang tien|"
            r"phí ship cao|phi ship cao|phí vận chuyển cao|phi van chuyen cao|"
            r"giá quá cao|gia qua cao|giá hơi cao|gia hoi cao|"
            r"giá khá cao|gia kha cao|giá hơi đắt|gia hoi dat|"
            r"giá khá đắt|gia kha dat)\b"
        ),
        "mention_blockers": [
            r"\b(?:đánh giá cao|danh gia cao|giá thể|gia the|bán đắt hàng|ban dat hang|"
            r"chúc .{0,20}đắt hàng|chuc .{0,20}dat hang)\b"
        ],
    },
}


def normalize_review_text(value: object) -> str:
    """Normalize review text while preserving Vietnamese diacritics."""
    text = unicodedata.normalize("NFKC", str(value or "").lower())
    return re.sub(r"\s+", " ", text).strip()


def mask_patterns(text: str, patterns: list[str] | None) -> str:
    masked = text
    for pattern in patterns or []:
        masked = re.sub(pattern, lambda m: " " * len(m.group(0)), masked)
    return masked


def mask_spans(text: str, matches: list[re.Match[str]]) -> str:
    """Mask match spans while preserving offsets for a parallel context string."""
    chars = list(text)
    for match in matches:
        chars[match.start() : match.end()] = " " * (match.end() - match.start())
    return "".join(chars)


def context_for_aspect(
    text: str,
    mention_pattern: str,
    mention_blockers: list[str] | None = None,
    radius: int = 90,
) -> str:
    mention_text = mask_patterns(text, mention_blockers)
    matches = list(re.finditer(mention_pattern, mention_text))
    if not matches:
        return ""
    windows = [mention_text[max(0, m.start() - radius) : m.end() + radius] for m in matches]
    return " | ".join(windows)


def classify_aspect(text_normalized: str, rules: dict[str, object]) -> tuple[int, str]:
    context = context_for_aspect(
        text_normalized,
        str(rules["mention"]),
        rules.get("mention_blockers"),
    )
    if not context:
        return 0, ""
    negative_context = mask_patterns(context, rules.get("negative_blockers"))
    negative_matches = list(re.finditer(str(rules["negative"]), negative_context))
    negative = bool(negative_matches)
    positive_context = mask_spans(context, negative_matches)
    positive = bool(re.search(str(rules["positive"]), positive_context))
    if positive and negative:
        return 1, "Mixed"
    if negative:
        return 1, "Negative"
    if positive:
        return 1, "Positive"
    neutral_override = rules.get("neutral_override")
    if neutral_override and re.search(str(neutral_override), context):
        return 1, "Neutral"
    return 1, "Neutral"

In [ ]:
# 5b) Áp dụng sentiment cho toàn bộ review
reviews["review_text_normalized"] = reviews["review_text"].map(normalize_review_text)

for aspect, rules in ASPECTS.items():
    slug = ASPECT_SLUGS[aspect]
    results = reviews["review_text_normalized"].map(
        lambda text: classify_aspect(text, rules)
    )
    reviews[f"{slug}_mention"] = [result[0] for result in results]
    reviews[f"{slug}_sentiment"] = [result[1] for result in results]
    print(f"Đã chạy: {aspect}")

reviews["sentiment_rule_version"] = RULE_VERSION


def overall_rule_sentiment(row: pd.Series) -> str:
    labels = [
        row[f"{slug}_sentiment"]
        for slug in ASPECT_SLUGS.values()
        if row[f"{slug}_mention"] == 1
    ]
    if not labels:
        return "Unclassified"
    has_positive = "Positive" in labels
    has_negative = "Negative" in labels
    has_mixed = "Mixed" in labels
    if has_mixed or (has_positive and has_negative):
        return "Mixed"
    if has_negative:
        return "Negative"
    if has_positive:
        return "Positive"
    return "Neutral"


reviews["overall_rule_sentiment"] = reviews.apply(overall_rule_sentiment, axis=1)


def wilson_interval(successes: int, trials: int, z: float = 1.96):
    if trials <= 0:
        return (np.nan, np.nan)
    rate = successes / trials
    denominator = 1 + (z * z / trials)
    center = (rate + z * z / (2 * trials)) / denominator
    margin = (
        z
        * math.sqrt((rate * (1 - rate) / trials) + (z * z / (4 * trials * trials)))
        / denominator
    )
    return (max(0.0, center - margin), min(1.0, center + margin))


aspect_rows = []
long_rows = []
for shop_name, shop_reviews in reviews.groupby("shop_name", sort=True):
    for aspect, slug in ASPECT_SLUGS.items():
        mentioned = shop_reviews.loc[shop_reviews[f"{slug}_mention"].eq(1)]
        counts = mentioned[f"{slug}_sentiment"].value_counts()
        positive = int(counts.get("Positive", 0))
        negative = int(counts.get("Negative", 0))
        mixed = int(counts.get("Mixed", 0))
        neutral = int(counts.get("Neutral", 0))
        negative_signal = negative + mixed
        ci_low, ci_high = wilson_interval(negative_signal, len(mentioned))
        aspect_rows.append({
            "shop_name": shop_name,
            "aspect": aspect,
            "clean_text_reviews": len(shop_reviews),
            "mention_count": len(mentioned),
            "mention_rate": len(mentioned) / len(shop_reviews) if len(shop_reviews) else np.nan,
            "positive_count": positive,
            "negative_count": negative,
            "mixed_count": mixed,
            "neutral_count": neutral,
            "negative_signal_count": negative_signal,
            "negative_signal_rate": negative_signal / len(mentioned) if len(mentioned) else np.nan,
            "negative_signal_ci95_low": ci_low,
            "negative_signal_ci95_high": ci_high,
            "rule_version": RULE_VERSION,
        })
        for _, row in mentioned.iterrows():
            long_rows.append({
                "review_id_v2": row["review_id_v2"],
                "shop_name": shop_name,
                "aspect": aspect,
                "sentiment": row[f"{slug}_sentiment"],
                "review_text": row["review_text"],
                "rating_star": row.get("rating_star", ""),
                "review_date": row.get("review_date", ""),
            })

aspect_summary_by_shop = pd.DataFrame(aspect_rows)
aspect_long = pd.DataFrame(long_rows)
display(aspect_summary_by_shop)


## Phần B — Topic modeling TF-IDF + NMF

In [ ]:
# Cấu hình NMF cố định để tái lập topic
N_TOPICS = 10
TOP_TERMS = 15
REPRESENTATIVE_REVIEWS = 3
MAX_FEATURES = 15_000
MIN_DF = 10
MAX_DF = 0.70
MIN_INFORMATIVE_TOKENS = 3
FIT_MAX_PER_SHOP = 6_000
COMMON_WINDOW_START = pd.Timestamp("2025-07-17")
COMMON_WINDOW_END = pd.Timestamp("2026-07-16")

print({
    "n_topics": N_TOPICS,
    "fit_max_per_shop": FIT_MAX_PER_SHOP,
    "random_seed": RANDOM_SEED,
})


In [ ]:
# 6a) Làm sạch text riêng cho topic modeling
VI_STOPWORDS = set(
    """
    a ai anh bạn bằng bị bởi cái các cách cho chỉ chị có của cũng đã đang đây để
    đến được gì gửi hay khi là làm lại lên mà mình một này nếu như những nó ở
    qua ra rằng rồi sau sẽ shop sản phẩm hàng mua dùng sử dụng thấy thì trong
    trên trước từ tôi tụi và vào về với vẫn vậy vì quá rất nhé nha luôn lần
    mọi người em mình mình ạ ơi đây kia đó này nọ
    """.split()
)

GENERIC_TOKENS = set(
    """
    ok oke okay tốt tot ổn on được duoc nhận nhan xu đánh danh giá gia hình hinh
    ảnh anh video chất chat lượng luong sản san phẩm pham hàng hang shop
    """.split()
)

BOILERPLATE_PATTERNS = [
    r"\b(?:đánh giá|danh gia).{0,40}(?:nhận xu|nhan xu)\b",
    r"\b(?:hình ảnh|hinh anh|video).{0,50}(?:nhận xu|nhan xu)\b",
    r"\b(?:nhận xu|nhan xu)\b",
    r"\b(?:hình ảnh|hinh anh).{0,30}(?:mang tính chất|mang tinh chat)\b",
]

TEENCODE = {
    "ko": "không",
    "k": "không",
    "kg": "không",
    "khg": "không",
    "đc": "được",
    "dc": "được",
    "sp": "sản phẩm",
}


def topic_clean_text(value: object) -> str:
    text = unicodedata.normalize("NFKC", str(value or "").lower())
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    for pattern in BOILERPLATE_PATTERNS:
        text = re.sub(pattern, " ", text)
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"[_\d]+", " ", text)
    tokens = [TEENCODE.get(token, token) for token in text.split()]
    return " ".join(tokens)


def informative_token_count(text: str) -> int:
    tokens = [
        token
        for token in text.split()
        if token not in VI_STOPWORDS and token not in GENERIC_TOKENS and len(token) > 1
    ]
    return len(tokens)


topic_documents = reviews[
    ["review_id_v2", "shop_name", "review_text", "review_date_parsed", "overall_rule_sentiment"]
].copy()
topic_documents["topic_text"] = topic_documents["review_text"].map(topic_clean_text)
topic_documents["informative_tokens"] = topic_documents["topic_text"].map(
    informative_token_count
)
topic_documents["topic_eligible"] = topic_documents["informative_tokens"].ge(
    MIN_INFORMATIVE_TOKENS
)

eligible = topic_documents.loc[topic_documents["topic_eligible"]].copy()
if len(eligible) < max(100, N_TOPICS * 10):
    raise ValueError(
        f"Chỉ có {len(eligible)} review đủ thông tin; không đủ để fit {N_TOPICS} topic."
    )

fit_pieces = []
for index, (shop, group) in enumerate(eligible.groupby("shop_name", sort=True)):
    sample_n = min(FIT_MAX_PER_SHOP, len(group))
    fit_pieces.append(group.sample(n=sample_n, random_state=RANDOM_SEED + index))
fit_documents = (
    pd.concat(fit_pieces, ignore_index=True)
    .drop_duplicates(subset=["topic_text"])
    .reset_index(drop=True)
)

print({
    "all_reviews": len(topic_documents),
    "topic_eligible": len(eligible),
    "low_information": int((~topic_documents["topic_eligible"]).sum()),
    "fit_unique_balanced": len(fit_documents),
})
display(
    fit_documents.groupby("shop_name", as_index=False)
    .agg(fit_documents=("review_id_v2", "size"))
)


In [ ]:
# 6b) Fit TF–IDF + NMF và transform toàn bộ review đủ thông tin
effective_min_df = 2 if len(fit_documents) < 1_000 else MIN_DF
vectorizer = TfidfVectorizer(
    lowercase=False,
    stop_words=sorted(VI_STOPWORDS),
    ngram_range=(1, 2),
    min_df=effective_min_df,
    max_df=MAX_DF,
    max_features=MAX_FEATURES,
    sublinear_tf=True,
    norm="l2",
    dtype=np.float32,
    token_pattern=r"(?u)\b[^\W\d_][^\W\d_]+\b",
)
X_fit = vectorizer.fit_transform(fit_documents["topic_text"])
actual_topics = min(N_TOPICS, X_fit.shape[0] - 1, X_fit.shape[1] - 1)
if actual_topics < 2:
    raise ValueError(f"Ma trận TF–IDF quá nhỏ để fit NMF: {X_fit.shape}")

nmf = NMF(
    n_components=actual_topics,
    init="nndsvda",
    solver="cd",
    beta_loss="frobenius",
    max_iter=500,
    tol=1e-4,
    random_state=RANDOM_SEED,
)
W_fit = nmf.fit_transform(X_fit)
H = nmf.components_
X_all = vectorizer.transform(eligible["topic_text"])
W_all = nmf.transform(X_all)

if not np.isfinite(W_all).all() or (W_all < 0).any():
    raise AssertionError("NMF weights có NaN/Inf hoặc số âm.")

# Chuẩn hóa scale W–H trước khi tính topic_share; topic_share không phải xác suất.
component_scale = H.sum(axis=1)
component_scale = np.where(component_scale > 0, component_scale, 1.0)
W_scaled = W_all * component_scale[np.newaxis, :]
row_scale = W_scaled.sum(axis=1, keepdims=True)
topic_shares = np.divide(
    W_scaled,
    row_scale,
    out=np.zeros_like(W_scaled),
    where=row_scale > 0,
)

feature_names = np.asarray(vectorizer.get_feature_names_out())
keyword_rows = []
topic_labels = {}
for topic_index, component in enumerate(H):
    top_indices = component.argsort()[-TOP_TERMS:][::-1]
    top_terms = feature_names[top_indices]
    label = f"Topic {topic_index + 1:02d} — {', '.join(top_terms[:4])}"
    topic_labels[topic_index] = label
    for rank, term_index in enumerate(top_indices, start=1):
        keyword_rows.append({
            "topic_id": topic_index + 1,
            "auto_label": label,
            "term_rank": rank,
            "term": feature_names[term_index],
            "term_weight": float(component[term_index]),
        })

nmf_topic_keywords = pd.DataFrame(keyword_rows)

top1_index = topic_shares.argmax(axis=1)
sorted_index = np.argsort(topic_shares, axis=1)
top2_index = sorted_index[:, -2] if actual_topics >= 2 else top1_index
top1_share = topic_shares[np.arange(len(topic_shares)), top1_index]
top2_share = topic_shares[np.arange(len(topic_shares)), top2_index]

eligible["topic_1_id"] = top1_index + 1
eligible["topic_1_label"] = [topic_labels[index] for index in top1_index]
eligible["topic_1_share"] = top1_share
eligible["topic_2_id"] = top2_index + 1
eligible["topic_2_label"] = [topic_labels[index] for index in top2_index]
eligible["topic_2_share"] = top2_share
eligible["topic_margin"] = top1_share - top2_share
eligible["topic_status"] = np.where(row_scale.ravel() > 0, "ASSIGNED", "UNASSIGNED")

topic_assignment_columns = [
    "review_id_v2",
    "topic_1_id",
    "topic_1_label",
    "topic_1_share",
    "topic_2_id",
    "topic_2_label",
    "topic_2_share",
    "topic_margin",
    "topic_status",
]
reviews = reviews.merge(
    eligible[topic_assignment_columns],
    on="review_id_v2",
    how="left",
    validate="one_to_one",
)
reviews["topic_status"] = reviews["topic_status"].fillna("LOW_INFORMATION")

print({
    "tfidf_shape_fit": X_fit.shape,
    "tfidf_shape_all": X_all.shape,
    "n_topics": actual_topics,
    "nmf_iterations": int(nmf.n_iter_),
    "reconstruction_error_fit": float(nmf.reconstruction_err_),
    "assigned": int(reviews["topic_status"].eq("ASSIGNED").sum()),
    "low_information": int(reviews["topic_status"].eq("LOW_INFORMATION").sum()),
    "unassigned_zero_vector": int(reviews["topic_status"].eq("UNASSIGNED").sum()),
})
display(
    nmf_topic_keywords.pivot(
        index=["topic_id", "auto_label"], columns="term_rank", values="term"
    ).reset_index()
)


In [ ]:
# Đặt tên topic thủ công từ keywords và review đại diện
TOPIC_NAME_MANUAL = {
    1: "Đóng gói cẩn thận",
    2: "Chưa sử dụng / chưa thể đánh giá",
    3: "Chất lượng sản phẩm được khen",
    4: "Sản phẩm đúng mô tả",
    5: "Ý định mua lại và ủng hộ shop",
    6: "Giao hàng nhanh",
    7: "Hiệu quả sử dụng và kiểm soát sâu hại",
    8: "Độ chắc chắn của đóng gói / sản phẩm",
    9: "Trải nghiệm nông nghiệp và phản hồi hỗn hợp",
    10: "Giao đúng và đủ số lượng",
}
TOPIC_INTERPRETATION_CONFIDENCE = {
    1: "Cao", 2: "Cao", 3: "Cao", 4: "Cao", 5: "Cao",
    6: "Cao", 7: "Cao", 8: "Trung bình", 9: "Thấp", 10: "Cao",
}

representative_rows = []
eligible_reset = eligible.reset_index(drop=True)
for topic_index in range(actual_topics):
    order = np.argsort(topic_shares[:, topic_index])[-3:][::-1]
    row = {
        "topic_id": topic_index + 1,
        "topic_name_manual": TOPIC_NAME_MANUAL[topic_index + 1],
        "interpretation_confidence": TOPIC_INTERPRETATION_CONFIDENCE[topic_index + 1],
    }
    terms = (
        nmf_topic_keywords.loc[
            (nmf_topic_keywords["topic_id"] == topic_index + 1)
            & (nmf_topic_keywords["term_rank"] <= 10)
        ]
        .sort_values("term_rank")["term"]
        .tolist()
    )
    row["top_keywords"] = ", ".join(terms)
    for rank, row_index in enumerate(order, start=1):
        row[f"representative_example_{rank}"] = eligible_reset.iloc[row_index]["review_text"]
    representative_rows.append(row)

topic_definitions = pd.DataFrame(representative_rows)
display(topic_definitions)


In [ ]:
# Khóa tập diễn giải và Validation 100 của snapshot chính thức
OFFICIAL_TOPIC_DEFINITIONS = [{'topic_id': 1, 'topic_name_manual': 'Đóng gói cẩn thận', 'interpretation_confidence': 'Cao', 'top_keywords': 'cẩn, cẩn thận, thận, gói cẩn, gói, đóng, đóng gói, nhanh đóng, thận giao, thận chất', 'representative_example_1': 'Đóng gói cẩn thận', 'representative_example_2': 'Shop Đóng gói cẩn thận', 'representative_example_3': 'Đóng gói cẩn thận'}, {'topic_id': 2, 'topic_name_manual': 'Chưa sử dụng / chưa thể đánh giá', 'interpretation_confidence': 'Cao', 'top_keywords': 'chưa, biết, chưa biết, nên chưa, nên, chưa nên, nào, lượng chưa, thế nào, thế', 'representative_example_1': 'Chưa gieo nên chưa biết ntn', 'representative_example_2': 'Chưa gieo chưa biết', 'representative_example_3': 'Mới sử dụng chưa biết thế nào'}, {'topic_id': 3, 'topic_name_manual': 'Chất lượng sản phẩm được khen', 'interpretation_confidence': 'Cao', 'top_keywords': 'chất lượng, chất, lượng, lượng tốt, tốt, ok, lượng ok, tốt giao, nhanh chất, lượng chất', 'representative_example_1': 'Chất lượng sản phẩm:chất lượng tốt Mua dùm, quên chụp lại, hình ảnh video mang tính chất nhận xu', 'representative_example_2': 'Chất lượng sản phẩm:tuyet voii Đây là một sp đáng mua kavsodh akshsk aohjs aihsj síh', 'representative_example_3': 'Chất lượng:tôt Rễ mập mạp:đat'}, {'topic_id': 4, 'topic_name_manual': 'Sản phẩm đúng mô tả', 'interpretation_confidence': 'Cao', 'top_keywords': 'mô, tả, mô tả, đúng mô, đúng, giống mô, tả giao, giống, tả chất, lượng đúng', 'representative_example_1': 'Hàng đúng mô tả', 'representative_example_2': 'Đúng mô tả', 'representative_example_3': 'Sản phẩm đúng mô tả'}, {'topic_id': 5, 'topic_name_manual': 'Ý định mua lại và ủng hộ shop', 'interpretation_confidence': 'Cao', 'top_keywords': 'ủng, hộ, ủng hộ, tiếp, hộ tiếp, tục, tiếp tục, tục ủng, tốt ủng, tốt', 'representative_example_1': 'Sẽ ủng hộ lan sau', 'representative_example_2': 'Lần sau tiếp tục ủng hộ shop', 'representative_example_3': 'Lần sau sẽ ủng hộ shop tiếp'}, {'topic_id': 6, 'topic_name_manual': 'Giao hàng nhanh', 'interpretation_confidence': 'Cao', 'top_keywords': 'giao nhanh, nhanh, giao, chóng, nhanh chóng, tình, nhiệt, nhiệt tình, nhanh đúng, tốt giao', 'representative_example_1': 'Giao hàng nhanh! Thank shop', 'representative_example_2': 'Giao hàng nhanh, sản phẩm okie', 'representative_example_3': 'Giao hàng nhanh. Đã mua của shop nhiêu lần. Hình ảnh chỉ mang tính chất nhận xu.'}, {'topic_id': 7, 'topic_name_manual': 'Hiệu quả sử dụng và kiểm soát sâu hại', 'interpretation_confidence': 'Cao', 'top_keywords': 'quả, hiệu, hiệu quả, biết hiệu, kiến, quả không, không, kết, kết quả, quả tốt', 'representative_example_1': 'Diệt kiến hiệu quả', 'representative_example_2': 'Rất hiệu quả, 100% diệt sạch kiến lửa', 'representative_example_3': 'Hiệu quả:bot phan kien rat hieu qua rắt lên là kiên chết hêt luôn lân sau sẽ mua lai đê diệt kieến'}, {'topic_id': 8, 'topic_name_manual': 'Độ chắc chắn của đóng gói / sản phẩm', 'interpretation_confidence': 'Trung bình', 'top_keywords': 'chắn, chắc, chắc chắn, gói chắc, đóng gói, đóng, gói, chắn giao, đẹp, nhanh đóng', 'representative_example_1': 'Xẻng chắc chắn, sáng bóng.', 'representative_example_2': 'Đóng gói chắc chắn, hình ảnh chỉ mang tính chất nhận xu . 👍🏻👍🏻👍🏻☺️☺️☺️☺️', 'representative_example_3': 'Shop đóng gói sản phẩm chắc chắn'}, {'topic_id': 9, 'topic_name_manual': 'Trải nghiệm nông nghiệp và phản hồi hỗn hợp', 'interpretation_confidence': 'Thấp', 'top_keywords': 'không, giá, cây, quảng, quảng cáo, hạt, giống, nhiều, phân', 'representative_example_1': 'Mình mua của bên các bạn nhiều lần rồi, từ đất đến phân. Đất có cả xơ dừa tơi nhá', 'representative_example_2': 'Đất trồng tơi', 'representative_example_3': 'Đất dinh dưỡng nhưng thấy hơi tơi, khô, nhiều trấu, giữ nước hơi kém, không thích loại này lắm.'}, {'topic_id': 10, 'topic_name_manual': 'Giao đúng và đủ số lượng', 'interpretation_confidence': 'Cao', 'top_keywords': 'đủ, số, số lượng, đủ số, đúng, đầy đủ, đầy, giao đúng, giao, lượng', 'representative_example_1': 'Giao đúng và đầy đủ số lượng.', 'representative_example_2': 'Sl sp đầy đủ.', 'representative_example_3': 'Giao hàng đầy đủ số lượng, bán hàng có tâm uy tín. Chỗ tôgs để mua'}]
OFFICIAL_VALIDATION_REVIEW_IDS = ['a76bcc2bdf27864d7a6f27ef', '7c7aeb32878e4819cd9b9b87', 'bd8d6b6457893b6cc873ca2d', '4606245198fbf099c8e4e464', '4acbf59d200e73402b1f882b', '402f575d7bb0e123bb65f677', 'ebe1854643ebdcbcfc2552d4', '86f9a4b36d267dade1a5cee6', '2fcfbd0cda5a64a1de2ca92c', '36a90868cfca90e9d295d774', '4dd736244834948eb75d4033', '08af39357d7f9bcbf6e61b90', '2a9514777bc4d69c9607f040', 'ac558f66bbb92107c94250f7', '9a3a96bd96c059fa47aa982b', 'bd1da191264c89b6b6a99f13', '645c03a6333e7ff61423ae90', '04ae788ac046ffd3f53a2ec4', 'b55439709e1ce61c5d5dfa9b', '28331a91d2a4f58f61cf2859', '5f2ad194e24bc4800ff7e703', '76f23d829c03d9f2961b63a8', 'ca6c4fe238ae8dbeb86691d2', '1895b3aebf50861777b529fa', '30d697967543b8ebf40721d9', 'a7b175aaf7cf3cc28526742b', 'b1e221928e7e8944302a9bd3', 'e2195e839ad0b79fa9d7df9d', '1b75ab6a03df13f51118dd33', '6c9048f1432ec6adeaa8581d', 'd57ff76e280a9fce080234c1', '6694c00961df1b2cc00f35a0', '498fdba4a77efbdc04b80189', 'b6cf92bffbf8efb00adf6cbf', 'f168f534e03613990eb91690', 'db924e2029add191813c87ed', 'b6ee3a0ef4760b5fef6c7e2c', '63e381e1ea4faa38bdb024ea', 'dfea4f69452ce9cc26185866', '93a8785c67add4b80f690a69', '9ccfb0bf04cf38bb432ae6a0', '5db77bdfdd769e8686c7a5c6', '865cc9827a7a79dc19685172', '539986d477dcbb205d7602f4', '639e6b4e132bd3750fa7f8e8', '195e3e3df430c460dafea549', '0a9fa92ad5d1b212225631f5', '321699b8fe247704265e5acf', 'fc8db165c40ed3696a474198', '647e2ca8f2687e9105f492d5', '4eb6e762c00b169ac949dfe4', '45a4c0a2c2e81affee100130', '6f9cef4cb0eec43de93b72c2', '5c77a8ca59919985c3c535af', 'f039fe26d70f3a9076b6050b', '6228717ac3641b6d94f10e97', '1386998280900f57fa0215d3', '48992d91ac08f020a96166de', 'c0818b6d3c16e1d1f7df19bb', 'a1bfe619a20a73b3159ac739', '16799ac2a1cfe851c63ca5da', '8d3307f607400cf05928ed38', '84cea9a0b9f1d52f93294900', 'ef4fa736b187c42e923429ac', '5dd0193dea91af06477f65d7', '2b13bf7dc73af9e138c7d82e', 'fd642da8282b4c92b1bd76b9', '012ef6d07cb35abe87946500', '0b5cd9f11a370f28dab57063', '77b1ae40d83cbdd3c4df5078', '68e8c627b8d6f7cfdddb6b3f', 'bfa0ac17e3b65faa715e905d', 'c865ef3c7823c2c6d18dd77a', '4e62a0bf2d0571e2591e2bdb', '69ebc283f924172da89baf56', 'f5d6e62c86a6897792cf347a', 'e9368b5e11d9147689fcb51b', 'b74716c6cd25c73136989b1b', '2a146fd3bd5001d4fc5e39e2', 'd40392fe4a1bd747ed0f38ef', '6921a45b04ade28cc2b2dfd8', '65786b4d223ea1b5ecdf9c68', 'e092f511eb789b1574425369', 'c34f8061d14cb01313533831', '5b5f8534ce4abe94bfff8723', 'dc7984d0efce7ae88368d022', '7293138827283fc203e2de88', '97f82c0d1b58e52afaf8afb4', '06a8134c65cbcdf6e17bdee6', '939489c857ad4bd70e02487c', 'db442dc8c0d85511762e9756', 'a32a9e272f6334bda4ca8660', 'b363a79d2f8b034d2fe79ea1', 'f72c4a78857d26f88397ac6d', 'fcae4f40c7688e0785232a2f', '4ac3e59de5bcca44456e3157', '0e01bb4defa9413ccbeecaa3', '7b4f7474c32dca4251c5da79', '2deb5140136ae7a6bf6257ca', '3ebd4339079ff6b01a5267a4']

OFFICIAL_SNAPSHOT = (
    INPUT_SHA256 == EXPECTED_INPUT_SHA256
    and len(reviews) == 47_466
    and set(reviews["shop_name"].dropna().astype(str).unique()) ==
        {"An Việt", "Hiệp Thủy", "Hà Bắc", "Quang Anh VNUA", "Vườn Babylon"}
)
if OFFICIAL_SNAPSHOT:
    # Dùng đúng diễn giải đã chốt trong workbook sơ bộ để kết quả có thể tái lập.
    topic_definitions = pd.DataFrame(OFFICIAL_TOPIC_DEFINITIONS)
    print("Đã dùng Topic Definitions và Validation 100 đã khóa cho snapshot 47.466 review.")
else:
    print("Input khác snapshot chính thức: giữ topic definitions tính lại và lấy Validation 100 mới.")


## Phần C — Xuất workbook sơ bộ 6 sheet

In [ ]:
# Tạo đúng 6 bảng mục tiêu và xuất workbook Excel
SHOP_ORDER = ["An Việt", "Hiệp Thủy", "Hà Bắc", "Quang Anh VNUA", "Vườn Babylon"]
LABEL_ORDER = ["Negative", "Neutral", "Positive", "Unclassified"]


def safe_column(frame: pd.DataFrame, name: str, default="") -> pd.Series:
    if name in frame.columns:
        return frame[name]
    return pd.Series(default, index=frame.index)


# 1) Overall Sentiment
overall_rows = []
for shop in ["Tất cả shop", *SHOP_ORDER]:
    part = reviews if shop == "Tất cả shop" else reviews.loc[reviews["shop_name"].eq(shop)]
    counts = part["sentiment_label"].value_counts()
    total = len(part)
    valid = part.loc[part["sentiment_label"].ne("Unclassified")]
    overall_rows.append({
        "Shop": shop,
        "Negative": int(counts.get("Negative", 0)),
        "Neutral": int(counts.get("Neutral", 0)),
        "Positive": int(counts.get("Positive", 0)),
        "Unclassified": int(counts.get("Unclassified", 0)),
        "Total": total,
        "Positive share": counts.get("Positive", 0) / total if total else np.nan,
        "Negative share": counts.get("Negative", 0) / total if total else np.nan,
        "Mean confidence": valid["sentiment_confidence"].mean(),
    })
overall_table = pd.DataFrame(overall_rows)


# 2) Aspect Summary
overall_aspect_rows = []
for aspect in ASPECT_SLUGS:
    part = aspect_summary_by_shop.loc[aspect_summary_by_shop["aspect"].eq(aspect)]
    mentions = int(part["mention_count"].sum())
    positive = int(part["positive_count"].sum())
    negative_signal = int(part["negative_signal_count"].sum())
    overall_aspect_rows.append({
        "Aspect": aspect,
        "Clean reviews": len(reviews),
        "Mentions": mentions,
        "Mention rate": mentions / len(reviews) if len(reviews) else np.nan,
        "Positive": positive,
        "Positive rate": positive / mentions if mentions else np.nan,
        "Negative signal": negative_signal,
        "Negative-signal rate": negative_signal / mentions if mentions else np.nan,
    })
aspect_overall_full = pd.DataFrame(overall_aspect_rows)
summary_aspect_order = [
    "Tư vấn & dịch vụ",
    "Đóng gói",
    "Giao hàng",
    "Giá",
    "Chất lượng & hiệu quả",
]
aspect_summary_table = (
    aspect_overall_full.set_index("Aspect").loc[summary_aspect_order].reset_index()
)

gap_rows = []
for aspect in summary_aspect_order:
    aspect_part = aspect_summary_by_shop.loc[aspect_summary_by_shop["aspect"].eq(aspect)]
    hiepthuy = aspect_part.loc[aspect_part["shop_name"].eq("Hiệp Thủy")].iloc[0]
    competitors = aspect_part.loc[aspect_part["shop_name"].ne("Hiệp Thủy")]
    ht_rate = (
        hiepthuy["negative_signal_count"] / hiepthuy["mention_count"]
        if hiepthuy["mention_count"] else np.nan
    )
    comp_mentions = competitors["mention_count"].sum()
    comp_rate = (
        competitors["negative_signal_count"].sum() / comp_mentions
        if comp_mentions else np.nan
    )
    gap = ht_rate - comp_rate
    interpretation = (
        "Hiệp Thủy xấu hơn benchmark"
        if gap > 0.005
        else "Hiệp Thủy tốt hơn benchmark"
        if gap < -0.005
        else "Tương đương benchmark"
    )
    gap_rows.append({
        "Aspect": aspect,
        "Hiệp Thủy": ht_rate,
        "Competitors": comp_rate,
        "Gap (pp)": gap,
        "Interpretation": interpretation,
    })
aspect_gap_table = pd.DataFrame(gap_rows).sort_values("Gap (pp)", ascending=False)


# 3) Aspect by Shop — giữ 8 cột đúng workbook sơ bộ
aspect_by_shop_table = (
    aspect_summary_by_shop[
        [
            "shop_name", "aspect", "clean_text_reviews", "mention_count",
            "mention_rate", "positive_count", "negative_count", "neutral_count",
        ]
    ]
    .assign(
        shop_name=lambda frame: pd.Categorical(
            frame["shop_name"], categories=SHOP_ORDER, ordered=True
        ),
        aspect=lambda frame: pd.Categorical(
            frame["aspect"], categories=list(ASPECT_SLUGS), ordered=True
        ),
    )
    .sort_values(["shop_name", "aspect"])
    .reset_index(drop=True)
)
aspect_by_shop_table["shop_name"] = aspect_by_shop_table["shop_name"].astype(str)
aspect_by_shop_table["aspect"] = aspect_by_shop_table["aspect"].astype(str)


# 4) Validation 100 — giữ đúng mẫu đã khóa nếu là snapshot chính thức
if OFFICIAL_SNAPSHOT:
    validation_order = pd.DataFrame({
        "review_id_v2": OFFICIAL_VALIDATION_REVIEW_IDS,
        "_validation_order": np.arange(len(OFFICIAL_VALIDATION_REVIEW_IDS)),
    })
    validation_rows = (
        validation_order.merge(
            reviews[
                [
                    "review_id_v2", "shop_name", "rating_star",
                    "review_text", "sentiment_label",
                ]
            ],
            on="review_id_v2",
            how="left",
            validate="one_to_one",
        )
        .sort_values("_validation_order")
    )
    if validation_rows["review_text"].isna().any():
        raise AssertionError("Snapshot chính thức thiếu review trong Validation 100 đã khóa.")
else:
    validation_rows = (
        random_validation_100[["_source_row"]]
        .merge(
            reviews[
                ["_source_row", "shop_name", "rating_star", "review_text", "sentiment_label"]
            ],
            on="_source_row",
            how="left",
            validate="one_to_one",
        )
    )
validation_table = validation_rows[
    ["shop_name", "rating_star", "review_text", "sentiment_label"]
].copy()


# 5) Scored Reviews — đúng 13 cột của workbook mục tiêu
scored_reviews_table = pd.DataFrame({
    "review_id": reviews["review_id_v2"],
    "shop_name": safe_column(reviews, "shop_name"),
    "item_id": safe_column(reviews, "item_id"),
    "product_title_clean": safe_column(reviews, "product_title_clean"),
    "canonical_category": safe_column(reviews, "canonical_category"),
    "rating_star": safe_column(reviews, "rating_star"),
    "review_date": safe_column(reviews, "review_date"),
    "review_text": reviews["review_text"],
    "sentiment_text_clean": reviews["sentiment_text_clean"],
    "score_negative": reviews["score_negative"],
    "score_positive": reviews["score_positive"],
    "score_neutral": reviews["score_neutral"],
    "sentiment_label": reviews["sentiment_label"],
})


def excel_safe_text(value):
    if isinstance(value, str) and value.startswith(("+", "-", "@")):
        return "'" + value
    return value


for frame in [
    overall_table,
    aspect_summary_table,
    aspect_gap_table,
    aspect_by_shop_table,
    topic_definitions,
    validation_table,
    scored_reviews_table,
]:
    for column in frame.select_dtypes(include=["object"]).columns:
        frame[column] = frame[column].map(excel_safe_text)


OUTPUT_XLSX = OUTPUT_DIR / OUTPUT_NAME
with pd.ExcelWriter(
    OUTPUT_XLSX,
    engine="xlsxwriter",
    engine_kwargs={
        "options": {
            "strings_to_formulas": False,
            "strings_to_urls": False,
        }
    },
) as writer:
    workbook = writer.book
    navy = "#17365D"
    blue = "#1F4E78"
    pale_blue = "#D9EAF7"
    pale_yellow = "#FFF2CC"
    white = "#FFFFFF"
    green = "#E2F0D9"
    red = "#F4CCCC"

    title_fmt = workbook.add_format({
        "bold": True, "font_color": white, "bg_color": navy,
        "font_size": 16, "align": "left", "valign": "vcenter",
    })
    header_fmt = workbook.add_format({
        "bold": True, "font_color": white, "bg_color": blue,
        "border": 1, "border_color": "#9FBAD0",
        "text_wrap": True, "valign": "vcenter",
    })
    section_fmt = workbook.add_format({
        "bold": True, "font_color": navy, "bg_color": pale_blue,
        "border": 1, "border_color": "#9FBAD0",
    })
    percent_fmt = workbook.add_format({"num_format": "0.00%"})
    integer_fmt = workbook.add_format({"num_format": "#,##0"})
    wrap_fmt = workbook.add_format({"text_wrap": True, "valign": "top"})
    hiệp_thủy_fmt = workbook.add_format({"bg_color": green, "bold": True})

    # Overall Sentiment
    overall_table.to_excel(writer, sheet_name="Overall Sentiment", startrow=2, index=False)
    ws = writer.sheets["Overall Sentiment"]
    ws.hide_gridlines(2)
    ws.merge_range("A1:I2", "OVERALL SENTIMENT BY VISOBERT ", title_fmt)
    ws.set_row(2, 28)
    for col, value in enumerate(overall_table.columns):
        ws.write(2, col, value, header_fmt)
    ws.set_column("A:A", 22)
    ws.set_column("B:F", 14, integer_fmt)
    ws.set_column("G:I", 18, percent_fmt)
    ws.freeze_panes(3, 0)
    ws.autofilter(2, 0, 2 + len(overall_table), 8)
    ws.conditional_format(3, 0, 3 + len(overall_table) - 1, 8, {
        "type": "formula", "criteria": '=$A4="Hiệp Thủy"', "format": hiệp_thủy_fmt
    })
    chart = workbook.add_chart({"type": "column"})
    for column, color in [(1, "#C44E52"), (2, "#DDC36A"), (3, "#2F6B5F")]:
        chart.add_series({
            "name": ["Overall Sentiment", 2, column],
            "categories": ["Overall Sentiment", 3, 0, 3 + len(overall_table) - 1, 0],
            "values": ["Overall Sentiment", 3, column, 3 + len(overall_table) - 1, column],
            "fill": {"color": color},
        })
    chart.set_title({"name": "Sentiment count by shop"})
    chart.set_y_axis({"num_format": "#,##0"})
    chart.set_legend({"position": "bottom"})
    ws.insert_chart("K3", chart, {"x_scale": 1.2, "y_scale": 1.15})

    # Aspect Summary
    aspect_summary_table.to_excel(writer, sheet_name="Aspect Summary", startrow=3, index=False)
    aspect_gap_table.to_excel(
        writer, sheet_name="Aspect Summary", startrow=3, startcol=9, index=False
    )
    ws = writer.sheets["Aspect Summary"]
    ws.hide_gridlines(2)
    ws.merge_range("A1:N2", "ASPECT SENTIMENT — SUMMARY & HIỆP THỦY GAP", title_fmt)
    ws.merge_range(
        "A3:H3",
        "",
        section_fmt,
    )
    ws.merge_range("J3:N3", "HIỆP THỦY VS WEIGHTED COMPETITOR BENCHMARK", section_fmt)
    for col, value in enumerate(aspect_summary_table.columns):
        ws.write(3, col, value, header_fmt)
    for col, value in enumerate(aspect_gap_table.columns, start=9):
        ws.write(3, col, value, header_fmt)
    ws.set_column("A:A", 28)
    ws.set_column("B:C", 15, integer_fmt)
    ws.set_column("D:D", 16, percent_fmt)
    ws.set_column("E:E", 14, integer_fmt)
    ws.set_column("F:F", 16, percent_fmt)
    ws.set_column("G:G", 17, integer_fmt)
    ws.set_column("H:H", 20, percent_fmt)
    ws.set_column("I:I", 3)
    ws.set_column("J:J", 28)
    ws.set_column("K:M", 15, percent_fmt)
    ws.set_column("N:N", 32, wrap_fmt)
    ws.freeze_panes(4, 0)
    ws.conditional_format(4, 12, 4 + len(aspect_gap_table) - 1, 12, {
        "type": "cell", "criteria": ">", "value": 0.005,
        "format": workbook.add_format({"bg_color": red, "font_color": "#9C0006", "bold": True}),
    })
    ws.conditional_format(4, 12, 4 + len(aspect_gap_table) - 1, 12, {
        "type": "cell", "criteria": "<", "value": -0.005,
        "format": workbook.add_format({"bg_color": green, "font_color": "#274E13"}),
    })
    ws.write("K18", "Negative Signal rate = Negative signal/ Mentions of Aspect*100", wrap_fmt)
    chart = workbook.add_chart({"type": "column"})
    for column, color in [(10, "#2F6B5F"), (11, "#4C72B0")]:
        chart.add_series({
            "name": ["Aspect Summary", 3, column],
            "categories": ["Aspect Summary", 4, 9, 4 + len(aspect_gap_table) - 1, 9],
            "values": ["Aspect Summary", 4, column, 4 + len(aspect_gap_table) - 1, column],
            "fill": {"color": color},
        })
    chart.set_title({"name": "Negative-signal rate: Hiệp Thủy vs competitors"})
    chart.set_y_axis({"num_format": "0.0%"})
    chart.set_legend({"position": "bottom"})
    ws.insert_chart("A12", chart, {"x_scale": 1.25, "y_scale": 1.15})

    # Aspect by Shop
    aspect_by_shop_table.to_excel(writer, sheet_name="Aspect by Shop", startrow=3, index=False)
    ws = writer.sheets["Aspect by Shop"]
    ws.hide_gridlines(2)
    ws.merge_range("A1:H2", "ASPECT SENTIMENT BY SHOP — 30 SHOP × ASPECT ROWS", title_fmt)
    for col, value in enumerate(aspect_by_shop_table.columns):
        ws.write(3, col, value, header_fmt)
    ws.set_column("A:A", 21)
    ws.set_column("B:B", 29)
    ws.set_column("C:D", 16, integer_fmt)
    ws.set_column("E:E", 15, percent_fmt)
    ws.set_column("F:H", 15, integer_fmt)
    ws.freeze_panes(4, 2)
    ws.autofilter(3, 0, 3 + len(aspect_by_shop_table), 7)

    # Topic Definitions
    topic_definitions.to_excel(writer, sheet_name="Topic Definitions", startrow=2, index=False)
    ws = writer.sheets["Topic Definitions"]
    ws.hide_gridlines(2)
    ws.merge_range("A1:G2", "TOPIC DEFINITIONS ", title_fmt)
    for col, value in enumerate(topic_definitions.columns):
        ws.write(2, col, value, header_fmt)
    ws.set_column("A:A", 11, integer_fmt)
    ws.set_column("B:B", 40)
    ws.set_column("C:C", 22)
    ws.set_column("D:D", 55, wrap_fmt)
    ws.set_column("E:G", 58, wrap_fmt)
    ws.freeze_panes(3, 0)
    ws.autofilter(2, 0, 2 + len(topic_definitions), 6)

    # Validation 100
    validation_table.to_excel(writer, sheet_name="Validation 100", startrow=1, index=False)
    ws = writer.sheets["Validation 100"]
    ws.hide_gridlines(2)
    ws.merge_range("A1:D1", "Validation 100", title_fmt)
    for col, value in enumerate(validation_table.columns):
        ws.write(1, col, value, header_fmt)
    ws.set_column("A:A", 21)
    ws.set_column("B:B", 12)
    ws.set_column("C:C", 90, wrap_fmt)
    ws.set_column("D:D", 18)
    ws.freeze_panes(2, 0)
    ws.autofilter(1, 0, 1 + len(validation_table), 3)

    # Scored Reviews
    scored_reviews_table.to_excel(writer, sheet_name="Scored Reviews", startrow=2, index=False)
    ws = writer.sheets["Scored Reviews"]
    ws.hide_gridlines(2)
    ws.merge_range(
        "A1:M2",
        "SCORED REVIEWS ",
        title_fmt,
    )
    for col, value in enumerate(scored_reviews_table.columns):
        ws.write(2, col, value, header_fmt)
    widths = [27, 21, 16, 52, 22, 12, 14, 90, 90, 16, 16, 16, 18]
    for col, width in enumerate(widths):
        ws.set_column(col, col, width, wrap_fmt if col in (3, 7, 8) else None)
    ws.set_column("J:L", 16, workbook.add_format({"num_format": "0.000000"}))
    ws.freeze_panes(3, 2)
    ws.autofilter(2, 0, 2 + len(scored_reviews_table), 12)

print(f"Đã tạo workbook: {OUTPUT_XLSX}")
print({
    "overall_rows": len(overall_table),
    "aspect_summary_rows": len(aspect_summary_table),
    "aspect_by_shop_rows": len(aspect_by_shop_table),
    "topic_rows": len(topic_definitions),
    "validation_rows": len(validation_table),
    "scored_review_rows": len(scored_reviews_table),
})
display(overall_table)
display(aspect_summary_table)
display(aspect_gap_table)


In [ ]:
# QA workbook và tự tải Excel về
import openpyxl

EXPECTED_SHEETS = [
    "Overall Sentiment",
    "Aspect Summary",
    "Aspect by Shop",
    "Topic Definitions",
    "Validation 100",
    "Scored Reviews",
]
check_book = openpyxl.load_workbook(OUTPUT_XLSX, read_only=True, data_only=True)

qa_export = [
    ("official_input_sha256", INPUT_SHA256 == EXPECTED_INPUT_SHA256, INPUT_SHA256),
    ("official_snapshot_locked", OFFICIAL_SNAPSHOT, OFFICIAL_SNAPSHOT),
    ("fixed_output_name", OUTPUT_XLSX.name == OUTPUT_NAME, OUTPUT_XLSX.name),
    ("workbook_exists", OUTPUT_XLSX.exists(), str(OUTPUT_XLSX)),
    ("six_expected_sheets", check_book.sheetnames == EXPECTED_SHEETS, check_book.sheetnames),
    ("overall_total_reconciles", int(overall_table.iloc[0]["Total"]) == len(reviews), int(overall_table.iloc[0]["Total"])),
    ("sentiment_counts_reconcile", int(overall_table.iloc[0][LABEL_ORDER].sum()) == len(reviews), int(overall_table.iloc[0][LABEL_ORDER].sum())),
    ("aspect_shop_rows_30", len(aspect_by_shop_table) == 30, len(aspect_by_shop_table)),
    ("topics_10", len(topic_definitions) == 10, len(topic_definitions)),
    ("validation_100", len(validation_table) == min(100, int(reviews["sentiment_valid"].sum())), len(validation_table)),
    ("scored_rows_preserved", len(scored_reviews_table) == len(reviews), len(scored_reviews_table)),
    ("xlsx_scored_rows", check_book["Scored Reviews"].max_row == len(reviews) + 3, check_book["Scored Reviews"].max_row),
]

if len(reviews) == 47_466 and set(SHOP_ORDER) == set(reviews["shop_name"].unique()):
    expected_counts = {
        "Negative": 3204,
        "Neutral": 3023,
        "Positive": 41232,
        "Unclassified": 7,
    }
    observed_counts = {
        label: int(overall_table.iloc[0][label])
        for label in LABEL_ORDER
    }
    qa_export.append(
        ("official_snapshot_sentiment_counts", observed_counts == expected_counts, observed_counts)
    )
    expected_mentions = {
        "Tư vấn & dịch vụ": 1749,
        "Đóng gói": 10900,
        "Giao hàng": 17868,
        "Giá": 3386,
        "Chất lượng & hiệu quả": 22361,
    }
    observed_mentions = dict(
        zip(aspect_summary_table["Aspect"], aspect_summary_table["Mentions"].astype(int))
    )
    qa_export.append(
        ("official_snapshot_aspect_mentions", observed_mentions == expected_mentions, observed_mentions)
    )

qa_export_report = pd.DataFrame([
    {"check": name, "passed": bool(passed), "observed": str(observed)}
    for name, passed, observed in qa_export
])
display(qa_export_report)
if not qa_export_report["passed"].all():
    raise AssertionError(
        "QA export thất bại:\n"
        + qa_export_report.loc[~qa_export_report["passed"]].to_string(index=False)
    )

print(f"FINAL QA: PASS — {int(qa_export_report['passed'].sum())}/{len(qa_export_report)} checks")
print("KẾT QUẢ SƠ BỘ — chưa báo cáo accuracy/macro-F1 trước khi có nhãn người thật.")

if IN_COLAB:
    files.download(str(OUTPUT_XLSX))
else:
    print(f"File nằm tại: {OUTPUT_XLSX}")


## Checks & Handoff

Chỉ nộp hoặc sử dụng workbook được tải xuống khi cell cuối báo `FINAL QA: PASS`.  
File đầu ra: `MIS495_Thy_Hau_Sentiment_Reproduced.xlsx`.

Khi nộp cho giảng viên, gửi kèm notebook này và đúng file input `reviews_master_text_v1.csv`; checksum của input được notebook tự kiểm tra trước khi chạy phân tích.
